# E12 — safety projection

Extends Wang, Jacob, Kaushik & Zhang, arXiv:2603.07237.

**Run this after the main production run.** It is self-contained and independent of it.

## Why

Our runs show the learned agent breaking the upper voltage bound in every configuration,
while droop does not:

| controller | VphMax |
|---|---|
| Droop (constrained) | **1.050** |
| RL, E2 mild | 1.064 |
| RL, E4 paper-style | 1.067 |
| RL, E3 mild `w=100` | 1.107 |
| RL, E1 multi/mild (no EV) | **1.234** |

A projection layer that pushes the commanded action back into the voltage-feasible set is
standard practice in learning-based Volt-VAR control — SAVER, model-augmented safe Volt-VAR,
safety-constrained MARL — so a reviewer will ask why we do not have one. It also matters for
our own headline: `IntViol` is two-sided, so part of the RL penalty **is** that overvoltage.
Without this layer we cannot separate *"allocates worse"* from *"breaks the upper bound and
is charged for it"*.

## Two projections

| mode | what it does |
|---|---|
| `scale` | reduce P and Q together along the commanded ray — the plainest projection |
| `shed_q` | reduce **reactive first**, touching active only if shedding all Q is not enough |

`shed_q` is motivated by our own **E7**: at matched apparent power, reactive buys 1.25–1.49×
*less* voltage than active while costing the same stored energy, so reactive is the right
thing to give up first. A smoke test at aggressive load gives both projections `IntHi 0.0`
and `VphMax 1.050`, but `shed_q` retains **1683 kWh against `scale`'s 1251** — same safety,
35% more delivered energy.

Read `IntHi` (the overvoltage half of `IntViol`) as the check that projection worked, and
`IntLo` as the undervoltage performance that actually matters.


In [ ]:
# 0. dependencies (installs only what is missing)
# NOTE: the import name is `opendssdirect` but the PyPI distribution is `opendssdirect.py`.
import importlib, subprocess, sys

def ensure(mod, pkg):
    try:
        importlib.import_module(mod)
        return
    except ImportError:
        pass
    print(f"installing {pkg} ...")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-1500:]);  print(r.stderr[-1500:])
        raise SystemExit(f"pip install {pkg} failed. Install it manually, then re-run this cell.")
    importlib.invalidate_caches()
    importlib.import_module(mod)

for mod, pkg in [("opendssdirect", "opendssdirect.py"),
                 ("gymnasium", "gymnasium"),
                 ("stable_baselines3", "stable-baselines3"),
                 ("matplotlib", "matplotlib")]:
    ensure(mod, pkg)

import numpy, torch
from importlib.metadata import version, PackageNotFoundError
try:
    dssver = version("opendssdirect.py")
except PackageNotFoundError:
    dssver = "?"
print(f"numpy {numpy.__version__} | torch {torch.__version__} | opendssdirect.py {dssver}")
print("deps OK")

### Materialize the feeder and modules (self-contained)

In [ ]:
%%writefile ieee34_master.dss
! Standard (Mod 1) model of IEEE 34 Bus Test Feeder

! Note: Mod 2 better accounts for distributed load.

Clear
Set DefaultBaseFrequency=60

New object=circuit.ieee34-1
~ basekv=69 pu=1.05 angle=30 mvasc3=200000  !stiffen up a bit over DSS default

! Substation Transformer  -- Modification: Make source very stiff by defining a tiny leakage Z
New Transformer.SubXF Phases=3 Windings=2 Xhl=0.01    ! normally 8
~ wdg=1 bus=sourcebus conn=Delta kv=69    kva=25000   %r=0.0005   !reduce %r, too
~ wdg=2 bus=800       conn=wye   kv=24.9  kva=25000   %r=0.0005

! import line codes with phase impedance matrices
Redirect IEEELineCodes.dss   ! revised according to Later test feeder doc

! Lines
New Line.L1     Phases=3 Bus1=800.1.2.3  Bus2=802.1.2.3  LineCode=300  Length=2.58   units=kft
New Line.L2     Phases=3 Bus1=802.1.2.3  Bus2=806.1.2.3  LineCode=300  Length=1.73   units=kft
New Line.L3     Phases=3 Bus1=806.1.2.3  Bus2=808.1.2.3  LineCode=300  Length=32.23   units=kft
New Line.L4     Phases=1 Bus1=808.2      Bus2=810.2      LineCode=303  Length=5.804   units=kft
New Line.L5     Phases=3 Bus1=808.1.2.3  Bus2=812.1.2.3  LineCode=300  Length=37.5   units=kft
New Line.L6     Phases=3 Bus1=812.1.2.3  Bus2=814.1.2.3  LineCode=300  Length=29.73   units=kft
New Line.L7     Phases=3 Bus1=814r.1.2.3 Bus2=850.1.2.3  LineCode=301  Length=0.01   units=kft
New Line.L8     Phases=1 Bus1=816.1      Bus2=818.1      LineCode=302  Length=1.71   units=kft
New Line.L9     Phases=3 Bus1=816.1.2.3  Bus2=824.1.2.3  LineCode=301  Length=10.21   units=kft
New Line.L10    Phases=1 Bus1=818.1      Bus2=820.1      LineCode=302  Length=48.15   units=kft
New Line.L11    Phases=1 Bus1=820.1      Bus2=822.1      LineCode=302  Length=13.74   units=kft
New Line.L12    Phases=1 Bus1=824.2      Bus2=826.2      LineCode=303  Length=3.03   units=kft
New Line.L13    Phases=3 Bus1=824.1.2.3  Bus2=828.1.2.3  LineCode=301  Length=0.84   units=kft
New Line.L14    Phases=3 Bus1=828.1.2.3  Bus2=830.1.2.3  LineCode=301  Length=20.44   units=kft
New Line.L15    Phases=3 Bus1=830.1.2.3  Bus2=854.1.2.3  LineCode=301  Length=0.52   units=kft
New Line.L16    Phases=3 Bus1=832.1.2.3  Bus2=858.1.2.3  LineCode=301  Length=4.9   units=kft
New Line.L17    Phases=3 Bus1=834.1.2.3  Bus2=860.1.2.3  LineCode=301  Length=2.02   units=kft
New Line.L18    Phases=3 Bus1=834.1.2.3  Bus2=842.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L19    Phases=3 Bus1=836.1.2.3  Bus2=840.1.2.3  LineCode=301  Length=0.86   units=kft
New Line.L20    Phases=3 Bus1=836.1.2.3  Bus2=862.1.2.3  LineCode=301  Length=0.28   units=kft
New Line.L21    Phases=3 Bus1=842.1.2.3  Bus2=844.1.2.3  LineCode=301  Length=1.35   units=kft
New Line.L22    Phases=3 Bus1=844.1.2.3  Bus2=846.1.2.3  LineCode=301  Length=3.64   units=kft
New Line.L23    Phases=3 Bus1=846.1.2.3  Bus2=848.1.2.3  LineCode=301  Length=0.53   units=kft
New Line.L24    Phases=3 Bus1=850.1.2.3  Bus2=816.1.2.3  LineCode=301  Length=0.31   units=kft
New Line.L25    Phases=3 Bus1=852r.1.2.3 Bus2=832.1.2.3  LineCode=301  Length=0.01   units=kft

! 24.9/4.16 kV  Transformer
New Transformer.XFM1  Phases=3 Windings=2 Xhl=4.08
~ wdg=1 bus=832       conn=wye   kv=24.9  kva=500    %r=0.95
~ wdg=2 bus=888       conn=Wye   kv=4.16  kva=500    %r=0.95

New Line.L26    Phases=1 Bus1=854.2      Bus2=856.2      LineCode=303  Length=23.33   units=kft
New Line.L27    Phases=3 Bus1=854.1.2.3  Bus2=852.1.2.3  LineCode=301  Length=36.83   units=kft
! 9-17-10 858-864 changed to phase A per error report
New Line.L28    Phases=1 Bus1=858.1      Bus2=864.1      LineCode=303  Length=1.62   units=kft
New Line.L29    Phases=3 Bus1=858.1.2.3  Bus2=834.1.2.3  LineCode=301  Length=5.83   units=kft
New Line.L30    Phases=3 Bus1=860.1.2.3  Bus2=836.1.2.3  LineCode=301  Length=2.68   units=kft
New Line.L31    Phases=1 Bus1=862.2      Bus2=838.2      LineCode=304  Length=4.86   units=kft
New Line.L32    Phases=3 Bus1=888.1.2.3  Bus2=890.1.2.3  LineCode=300  Length=10.56   units=kft

! Capacitors
New Capacitor.C844      Bus1=844        Phases=3        kVAR=300        kV=24.9
New Capacitor.C848      Bus1=848        Phases=3        kVAR=450        kV=24.9

! Regulators - three independent phases
! Regulator 1
new transformer.reg1a phases=1 windings=2 bank=reg1 buses=(814.1 814r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1a transformer=reg1a winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1b phases=1 windings=2 bank=reg1 buses=(814.2 814r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1b transformer=reg1b winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6
new transformer.reg1c phases=1 windings=2 bank=reg1 buses=(814.3 814r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg1c transformer=reg1c winding=2 vreg=122 band=2 ptratio=120 ctprim=100 R=2.7 X=1.6

! Regulator 2
new transformer.reg2a phases=1 windings=2 bank=reg2 buses=(852.1 852r.1) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2a transformer=reg2a winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2b phases=1 windings=2 bank=reg2 buses=(852.2 852r.2) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2b transformer=reg2b winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5
new transformer.reg2c phases=1 windings=2 bank=reg2 buses=(852.3 852r.3) conns='wye wye' kvs="14.376 14.376" kvas="20000 20000" XHL=1
new regcontrol.creg2c transformer=reg2c winding=2 vreg=124 band=2 ptratio=120 ctprim=100 R=2.5 X=1.5

! spot loads
New Load.S860       Bus1=860   Phases=3 Conn=Wye   Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S840       Bus1=840   Phases=3 Conn=Wye   Model=5 kV= 24.900 kW=  27.0 kVAR=  21.0
New Load.S844       Bus1=844   Phases=3 Conn=Wye   Model=2 kV= 24.900 kW= 405.0 kVAR= 315.0

New Load.S848       Bus1=848   Phases=3 Conn=Delta Model=1 kV= 24.900 kW=  60.0 kVAR=  48.0
New Load.S830a      Bus1=830.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830b      Bus1=830.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.S830c      Bus1=830.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  25.0 kVAR=  10.0
New Load.S890       Bus1=890   Phases=3 Conn=Delta Model=5 kV=  4.160 kW= 450.0 kVAR= 225.0

! distributed loads
New Load.D802_806sb Bus1=802.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806rb Bus1=806.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  15.0 kVAR=   7.5
New Load.D802_806sc Bus1=802.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0
New Load.D802_806rc Bus1=806.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   7.0

New Load.D808_810sb Bus1=808.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0
New Load.D808_810rb Bus1=810.2 Phases=1 Conn=Wye   Model=4 kV= 14.376 kW=   8.0 kVAR=   4.0

New Load.D818_820sa Bus1=818.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5
New Load.D818_820ra Bus1=820.1 Phases=1 Conn=Wye   Model=2 kV= 14.376 kW=  17.0 kVAR=   8.5

New Load.D820_822sa Bus1=820.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0
New Load.D820_822ra Bus1=822.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  67.5 kVAR=  35.0

New Load.D816_824sb Bus1=816.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0
New Load.D816_824rb Bus1=824.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   2.5 kVAR=   1.0

New Load.D824_826sb Bus1=824.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_826rb Bus1=826.2 Phases=1 Conn=Wye   Model=5 kV= 14.376 kW=  20.0 kVAR=  10.0
New Load.D824_828sc Bus1=824.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D824_828rc Bus1=828.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D828_830sa Bus1=828.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5
New Load.D828_830ra Bus1=830.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   3.5 kVAR=   1.5

New Load.D854_856sb Bus1=854.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0
New Load.D854_856rb Bus1=856.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   2.0 kVAR=   1.0

New Load.D832_858sa Bus1=832.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858ra Bus1=858.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.5 kVAR=   1.5
New Load.D832_858sb Bus1=832.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858rb Bus1=858.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   1.0 kVAR=   0.5
New Load.D832_858sc Bus1=832.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5
New Load.D832_858rc Bus1=858.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   3.0 kVAR=   1.5

! 9-17-10 858-864 changed to phase A per error report
New Load.D858_864sb Bus1=858.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5
New Load.D858_864rb Bus1=864.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   1.0 kVAR=   0.5

New Load.D858_834sa Bus1=858.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834ra Bus1=834.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   2.0 kVAR=   1.0
New Load.D858_834sb Bus1=858.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834rb Bus1=834.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   7.5 kVAR=   4.0
New Load.D858_834sc Bus1=858.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5
New Load.D858_834rc Bus1=834.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   6.5 kVAR=   3.5

New Load.D834_860sa Bus1=834.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860ra Bus1=860.1.2 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=   8.0 kVAR=   4.0
New Load.D834_860sb Bus1=834.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860rb Bus1=860.2.3 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  10.0 kVAR=   5.0
New Load.D834_860sc Bus1=834.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5
New Load.D834_860rc Bus1=860.3.1 Phases=1 Conn=Delta Model=2 kV= 24.900 kW=  55.0 kVAR=  27.5

New Load.D860_836sa Bus1=860.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836ra Bus1=836.1.2 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  15.0 kVAR=   7.5
New Load.D860_836sb Bus1=860.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836rb Bus1=836.2.3 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=   5.0 kVAR=   3.0
New Load.D860_836sc Bus1=860.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0
New Load.D860_836rc Bus1=836.3.1 Phases=1 Conn=Delta Model=1 kV= 24.900 kW=  21.0 kVAR=  11.0

New Load.D836_840sa Bus1=836.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840ra Bus1=840.1.2 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=   9.0 kVAR=   4.5
New Load.D836_840sb Bus1=836.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5
New Load.D836_840rb Bus1=840.2.3 Phases=1 Conn=Delta Model=5 kV= 24.900 kW=  11.0 kVAR=   5.5

New Load.D862_838sb Bus1=862.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0
New Load.D862_838rb Bus1=838.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  14.0 kVAR=   7.0

New Load.D842_844sa Bus1=842.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5
New Load.D842_844ra Bus1=844.1 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=   4.5 kVAR=   2.5

New Load.D844_846sb Bus1=844.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846rb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  12.5 kVAR=   6.0
New Load.D844_846sc Bus1=844.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5
New Load.D844_846rc Bus1=846.3 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  10.0 kVAR=   5.5

New Load.D846_848sb Bus1=846.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5
New Load.D846_848rb Bus1=848.2 Phases=1 Conn=Wye   Model=1 kV= 14.376 kW=  11.5 kVAR=   5.5

! Script to revise Vminpu property on all loads to allow voltage to sag to 85% without switching
! to constant Z model
Load.s860.vminpu=.85
Load.s840.vminpu=.85
Load.s844.vminpu=.85
Load.s848.vminpu=.85
Load.s830a.vminpu=.85
Load.s830b.vminpu=.85
Load.s830c.vminpu=.85
Load.s890.vminpu=.85
Load.d802_806sb.vminpu=.85
Load.d802_806rb.vminpu=.85
Load.d802_806sc.vminpu=.85
Load.d802_806rc.vminpu=.85
Load.d808_810sb.vminpu=.85
Load.d808_810rb.vminpu=.85
Load.d818_820sa.vminpu=.85
Load.d818_820ra.vminpu=.85
Load.d820_822sa.vminpu=.85
Load.d820_822ra.vminpu=.85
Load.d816_824sb.vminpu=.85
Load.d816_824rb.vminpu=.85
Load.d824_826sb.vminpu=.85
Load.d824_826rb.vminpu=.85
Load.d824_828sc.vminpu=.85
Load.d824_828rc.vminpu=.85
Load.d828_830sa.vminpu=.85
Load.d828_830ra.vminpu=.85
Load.d854_856sb.vminpu=.85
Load.d854_856rb.vminpu=.85
Load.d832_858sa.vminpu=.85
Load.d832_858ra.vminpu=.85
Load.d832_858sb.vminpu=.85
Load.d832_858rb.vminpu=.85
Load.d832_858sc.vminpu=.85
Load.d832_858rc.vminpu=.85
Load.d858_864sb.vminpu=.85
Load.d858_864rb.vminpu=.85
Load.d858_834sa.vminpu=.85
Load.d858_834ra.vminpu=.85
Load.d858_834sb.vminpu=.85
Load.d858_834rb.vminpu=.85
Load.d858_834sc.vminpu=.85
Load.d858_834rc.vminpu=.85
Load.d834_860sa.vminpu=.85
Load.d834_860ra.vminpu=.85
Load.d834_860sb.vminpu=.85
Load.d834_860rb.vminpu=.85
Load.d834_860sc.vminpu=.85
Load.d834_860rc.vminpu=.85
Load.d860_836sa.vminpu=.85
Load.d860_836ra.vminpu=.85
Load.d860_836sb.vminpu=.85
Load.d860_836rb.vminpu=.85
Load.d860_836sc.vminpu=.85
Load.d860_836rc.vminpu=.85
Load.d836_840sa.vminpu=.85
Load.d836_840ra.vminpu=.85
Load.d836_840sb.vminpu=.85
Load.d836_840rb.vminpu=.85
Load.d862_838sb.vminpu=.85
Load.d862_838rb.vminpu=.85
Load.d842_844sa.vminpu=.85
Load.d842_844ra.vminpu=.85
Load.d844_846sb.vminpu=.85
Load.d844_846rb.vminpu=.85
Load.d844_846sc.vminpu=.85
Load.d844_846rc.vminpu=.85
Load.d846_848sb.vminpu=.85
Load.d846_848rb.vminpu=.85


! let the DSS estimate voltage bases automatically
Set VoltageBases = "69,24.9,4.16, .48"
CalcVoltageBases


In [ ]:
%%writefile IEEELineCodes.dss
! this file was corrected 9/16/2010 to match the values in Kersting's files



! These line codes are used in the 123-bus circuit

New linecode.1 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
!!!~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )
~ rmatrix = [0.086666667 | 0.029545455 0.088371212 | 0.02907197 0.029924242 0.087405303]
~ xmatrix = [0.204166667 | 0.095018939 0.198522727 | 0.072897727 0.080227273 0.201723485]
~ cmatrix = [2.851710072 | -0.920293787  3.004631862 | -0.350755566  -0.585011253 2.71134756]

New linecode.2 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0316143 0.0889665 | 0.0312137 0.0306264 0.088205 )
!!!~ xmatrix = (0.200783 | 0.0855879 0.204877 | 0.0935314 0.0760312 0.20744 )
!!!~ cmatrix = (3.15896 | -0.481416 2.8965 | -0.679335 -0.22313 2.90301 )
~ rmatrix = [0.088371212 | 0.02992424  0.087405303 | 0.029545455 0.02907197 0.086666667]
~ xmatrix = [0.198522727 | 0.080227273  0.201723485 | 0.095018939 0.072897727 0.204166667]
~ cmatrix = [3.004631862 | -0.585011253 2.71134756 | -0.920293787  -0.350755566  2.851710072]

New linecode.3 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0306264 0.088205 | 0.0316143 0.0312137 0.0901946 )
!!!~ xmatrix = (0.204877 | 0.0760312 0.20744 | 0.0855879 0.0935314 0.200783 )
!!!~ cmatrix = (2.8965 | -0.22313 2.90301 | -0.481416 -0.679335 3.15896 )

~ rmatrix = [0.087405303 | 0.02907197 0.086666667  | 0.029924242 0.029545455 0.088371212]
~ xmatrix = [0.201723485 | 0.072897727 0.204166667 | 0.080227273 0.095018939 0.198522727]
~ cmatrix = [2.71134756  | -0.350755566 2.851710072 | -0.585011253 -0.920293787 3.004631862]

New linecode.4 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0889665 | 0.0316143 0.0901946 | 0.0306264 0.0312137 0.088205 )
!!!~ xmatrix = (0.204877 | 0.0855879 0.200783 | 0.0760312 0.0935314 0.20744 )
!!!~ cmatrix = (2.8965 | -0.481416 3.15896 | -0.22313 -0.679335 2.90301 )
~ rmatrix = [0.087405303 | 0.029924242 0.088371212 | 0.02907197   0.029545455 0.086666667]
~ xmatrix = [0.201723485 | 0.080227273 0.198522727 | 0.072897727 0.095018939 0.204166667]
~ cmatrix = [2.71134756  | -0.585011253 3.004631862 | -0.350755566 -0.920293787 2.851710072]

New linecode.5 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.0901946 | 0.0312137 0.088205 | 0.0316143 0.0306264 0.0889665 )
!!!~ xmatrix = (0.200783 | 0.0935314 0.20744 | 0.0855879 0.0760312 0.204877 )
!!!~ cmatrix = (3.15896 | -0.679335 2.90301 | -0.481416 -0.22313 2.8965 )

~ rmatrix = [0.088371212  |  0.029545455  0.086666667  |  0.029924242  0.02907197  0.087405303]
~ xmatrix = [0.198522727  |  0.095018939  0.204166667  |  0.080227273  0.072897727  0.201723485]
~ cmatrix = [3.004631862  | -0.920293787  2.851710072  |  -0.585011253  -0.350755566  2.71134756]

New linecode.6 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 | 0.0312137 0.0316143 0.0901946 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 | 0.0935314 0.0855879 0.200783 )
!!!~ cmatrix = (2.90301 | -0.22313 2.8965 | -0.679335 -0.481416 3.15896 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303 | 0.029545455  0.029924242  0.088371212]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485 | 0.095018939  0.080227273  0.198522727]
~ cmatrix = [2.851710072 | -0.350755566  2.71134756 | -0.920293787  -0.585011253  3.004631862]
New linecode.7 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.8 nphases=2 BaseFreq=60 units=kft
!!!~ rmatrix = (0.088205 | 0.0306264 0.0889665 )
!!!~ xmatrix = (0.20744 | 0.0760312 0.204877 )
!!!~ cmatrix = (2.75692 | -0.326659 2.82313 )
~ rmatrix = [0.086666667 | 0.02907197  0.087405303]
~ xmatrix = [0.204166667 | 0.072897727  0.201723485]
~ cmatrix = [2.569829596 | -0.52995137  2.597460011]
New linecode.9 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.10 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.11 nphases=1 BaseFreq=60 units=kft
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.12 nphases=3 BaseFreq=60 units=kft
!!!~ rmatrix = (0.291814 | 0.101656 0.294012 | 0.096494 0.101656 0.291814 )
!!!~ xmatrix = (0.141848 | 0.0517936 0.13483 | 0.0401881 0.0517936 0.141848 )
!!!~ cmatrix = (53.4924 | 0 53.4924 | 0 0 53.4924 )
~ rmatrix = [0.288049242 | 0.09844697  0.29032197 | 0.093257576  0.09844697  0.288049242]
~ xmatrix = [0.142443182 | 0.052556818  0.135643939 | 0.040852273  0.052556818  0.142443182]
~ cmatrix = [33.77150149 | 0  33.77150149 | 0  0  33.77150149]

! These line codes are used in the 34-node test feeder

New linecode.300 nphases=3 basefreq=60   units=kft   ! ohms per 1000ft  Corrected 11/30/05
~ rmatrix = [0.253181818   |  0.039791667     0.250719697  |   0.040340909      0.039128788     0.251780303]  !ABC ORDER
~ xmatrix = [0.252708333   |  0.109450758     0.256988636  |   0.094981061      0.086950758     0.255132576]
~ CMATRIX = [2.680150309   | -0.769281006     2.5610381    |  -0.499507676     -0.312072984     2.455590387]
New linecode.301 nphases=3 basefreq=60   units=kft
~ rmatrix = [0.365530303   |   0.04407197      0.36282197   |   0.04467803       0.043333333     0.363996212]
~ xmatrix = [0.267329545   |   0.122007576     0.270473485  |   0.107784091      0.099204545     0.269109848] 
~ cmatrix = [2.572492163   |  -0.72160598      2.464381882  |  -0.472329395     -0.298961096     2.368881119]
New linecode.302 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.303 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.530208 )
~ xmatrix = (0.281345 )
~ cmatrix = (2.12257 )
New linecode.304 nphases=1 basefreq=60   units=kft
~ rmatrix = (0.363958 )
~ xmatrix = (0.269167 )
~ cmatrix = (2.1922 )


! This may be for the 4-node test feeder, but is not actually referenced.
!  instead, the 4Bus*.dss files all use the wiredata and linegeometry inputs
!  to calculate these matrices from physical data.

New linecode.400 nphases=3 BaseFreq=60
~ rmatrix = (0.088205 | 0.0312137 0.0901946 | 0.0306264 0.0316143 0.0889665 )
~ xmatrix = (0.20744 | 0.0935314 0.200783 | 0.0760312 0.0855879 0.204877 )
~ cmatrix = (2.90301 | -0.679335 3.15896 | -0.22313 -0.481416 2.8965 )

! These are for the 13-node test feeder

New linecode.601 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0674673 | 0.0312137 0.0654777 | 0.0316143 0.0306264 0.0662392 )
!!!~ xmatrix = (0.195204  | 0.0935314 0.201861 | 0.0855879 0.0760312 0.199298 )
!!!~ cmatrix = (3.32591   | -0.743055 3.04217 | -0.525237 -0.238111 3.03116 )
~ rmatrix = [0.065625    | 0.029545455  0.063920455  | 0.029924242  0.02907197  0.064659091]
~ xmatrix = [0.192784091 | 0.095018939  0.19844697   | 0.080227273  0.072897727  0.195984848]
~ cmatrix = [3.164838036 | -1.002632425  2.993981593 | -0.632736516  -0.372608713  2.832670203]
New linecode.602 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.144361 | 0.0316143 0.143133 | 0.0312137 0.0306264 0.142372 )
!!!~ xmatrix = (0.226028 | 0.0855879 0.230122 | 0.0935314 0.0760312 0.232686 )
!!!~ cmatrix = (3.01091  | -0.443561 2.77543  | -0.624494 -0.209615 2.77847 )
~ rmatrix = [0.142537879 | 0.029924242  0.14157197   | 0.029545455  0.02907197  0.140833333]
~ xmatrix = [0.22375     | 0.080227273  0.226950758  | 0.095018939  0.072897727  0.229393939]
~ cmatrix = [2.863013423 | -0.543414918  2.602031589 | -0.8492585  -0.330962141  2.725162768]
New linecode.603 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.254472 | 0.0417943 0.253371 )
!!!~ xmatrix = (0.259467 | 0.0912376 0.261431 )
!!!~ cmatrix = (2.54676  | -0.28882 2.49502 )
~ rmatrix = [0.251780303 | 0.039128788  0.250719697]
~ xmatrix = [0.255132576 | 0.086950758  0.256988636]
~ cmatrix = [2.366017603 | -0.452083836  2.343963508]
New linecode.604 nphases=2 BaseFreq=60
!!!~ rmatrix = (0.253371 | 0.0417943 0.254472 )
!!!~ xmatrix = (0.261431 | 0.0912376 0.259467 )
!!!~ cmatrix = (2.49502 | -0.28882 2.54676 )
~ rmatrix = [0.250719697 | 0.039128788   0.251780303]
~ xmatrix = [0.256988636  | 0.086950758  0.255132576]
~ cmatrix = [2.343963508 | -0.452083836 2.366017603]
New linecode.605 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.254428 )
!!!~ xmatrix = (0.259546 )
!!!~ cmatrix = (2.50575 )
~ rmatrix = [0.251742424]
~ xmatrix = [0.255208333]
~ cmatrix = [2.270366128]
New linecode.606 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.152193 | 0.0611362 0.15035 | 0.0546992 0.0611362 0.152193 )
!!!~ xmatrix = (0.0825685 | 0.00548281 0.0745027 | -0.00339824 0.00548281 0.0825685 )
!!!~ cmatrix = (72.7203 | 0 72.7203 | 0 0 72.7203 )
~ rmatrix = [0.151174242 | 0.060454545  0.149450758 | 0.053958333  0.060454545  0.151174242]
~ xmatrix = [0.084526515 | 0.006212121  0.076534091 | -0.002708333  0.006212121  0.084526515]
~ cmatrix = [48.67459408 | 0  48.67459408 | 0  0  48.67459408]
New linecode.607 nphases=1 BaseFreq=60
!!!~ rmatrix = (0.255799 )
!!!~ xmatrix = (0.092284 )
!!!~ cmatrix = (50.7067 )
~ rmatrix = [0.254261364]
~ xmatrix = [0.097045455]
~ cmatrix = [44.70661522]

! These are for the 37-node test feeder, all underground

New linecode.721 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0554906 | 0.0127467 0.0501597 | 0.00640446 0.0127467 0.0554906 )
!!!~ xmatrix = (0.0372331 | -0.00704588 0.0358645 | -0.00796424 -0.00704588 0.0372331 )
!!!~ cmatrix = (124.851 | 0 124.851 | 0 0 124.851 )
~ rmatrix = [0.055416667 | 0.012746212  0.050113636  | 0.006382576  0.012746212  0.055416667]
~ xmatrix = [0.037367424 | -0.006969697  0.035984848 | -0.007897727  -0.006969697  0.037367424]
~ cmatrix = [80.27484728 | 0  80.27484728            | 0  0  80.27484728]
New linecode.722 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.0902251 | 0.0309584 0.0851482 | 0.0234946 0.0309584 0.0902251 )
!!!~ xmatrix = (0.055991 | -0.00646552 0.0504025 | -0.0117669 -0.00646552 0.055991 )
!!!~ cmatrix = (93.4896 | 0 93.4896 | 0 0 93.4896 )
~ rmatrix = [0.089981061 | 0.030852273  0.085        | 0.023371212  0.030852273  0.089981061]
~ xmatrix = [0.056306818 | -0.006174242  0.050719697 | -0.011496212  -0.006174242  0.056306818]
~ cmatrix = [64.2184109 | 0  64.2184109              | 0  0  64.2184109]
New linecode.723 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.247572 | 0.0947678 0.249104 | 0.0893782 0.0947678 0.247572 )
!!!~ xmatrix = (0.126339 | 0.0390337 0.118816 | 0.0279344 0.0390337 0.126339 )
!!!~ cmatrix = (58.108 | 0 58.108 | 0 0 58.108 )
~ rmatrix = [0.245 | 0.092253788  0.246628788 | 0.086837121  0.092253788  0.245]
~ xmatrix = [0.127140152 | 0.039981061  0.119810606 | 0.028806818  0.039981061  0.127140152]
~ cmatrix = [37.5977112 | 0  37.5977112 | 0  0  37.5977112]
New linecode.724 nphases=3 BaseFreq=60
!!!~ rmatrix = (0.399883 | 0.101765 0.402011 | 0.0965199 0.101765 0.399883 )
!!!~ xmatrix = (0.146325 | 0.0510963 0.139305 | 0.0395402 0.0510963 0.146325 )
!!!~ cmatrix = (46.9685 | 0 46.9685 | 0 0 46.9685 )
~ rmatrix = [0.396818182 | 0.098560606  0.399015152 | 0.093295455  0.098560606  0.396818182]
~ xmatrix = [0.146931818 | 0.051856061  0.140113636 | 0.040208333  0.051856061  0.146931818]
~ cmatrix = [30.26701029 | 0  30.26701029 | 0  0  30.26701029]


In [ ]:
%%writefile v2g_sys.py
"""
System layer for the V2G gap study — paper-faithful corrections over v2g_core.py.

Differences from the original reproduction, all deliberate:
  * Feeder(control_mode=...)   : regulators can be STATIC (OpenDSS default, they act)
                                 or OFF (frozen). The paper never states which; E0 decides
                                 it empirically from their reported baseline fingerprint.
  * droop_pq(sat=0.90/1.10)    : matches the paper's stated saturation. The earlier code
                                 used 0.94/1.06, which made droop noticeably stronger.
  * per-PHASE voltages          : ANSI C84.1 limits are per-phase. We keep every energized
                                 node voltage instead of averaging phases per bus.
  * stochastic availability     : n_avail is a binomial draw around the mean profile, so
                                 scenario variance is explicit (and can be paired away).
  * throughput accounting       : cumulative battery kWh, for the degradation objective.
"""
import os
import numpy as np
import opendssdirect as dss

MASTER = os.path.abspath("ieee34_master.dss")

CFG = dict(
    v_min=0.95, v_max=1.05,
    hub_buses_multi=["890", "844", "832", "830", "860"],
    hub_bus_single=["890"],
    P_rated=500.0, Q_rated=400.0,               # kW / kVAr per hub
    ev_capacity=75.0, soc_init=0.7, soc_min=0.2, soc_max=0.9,
    soh=0.95, eta_inv=0.96, c_rate=0.5, n_ev=15,
    active_hours=list(range(6, 24)),            # 06:00 .. 23:00
    peak_mild=1.5, peak_aggr=3.0,
)

# Normalised daily load shape (peak 1.0 near 18:00).
LOAD_SHAPE = np.array([0.24, 0.22, 0.20, 0.20, 0.22, 0.26, 0.30, 0.42, 0.55, 0.63,
                       0.70, 0.74, 0.77, 0.79, 0.82, 0.86, 0.92, 0.97, 1.00, 0.96,
                       0.86, 0.66, 0.44, 0.30])

# Mean EV participation. Paper states 45-85% for the single-hub fleet.
AVAIL_MEAN = np.array([0.85, 0.85, 0.85, 0.85, 0.80, 0.75, 0.65, 0.55, 0.50, 0.47,
                       0.45, 0.45, 0.48, 0.50, 0.55, 0.60, 0.65, 0.72, 0.78, 0.82,
                       0.85, 0.85, 0.85, 0.85])

BASE_LL_KV = {"890": 4.16, "888": 4.16}   # 4.16 kV transformer secondary


def lam_profile(peak):
    return LOAD_SHAPE * peak


# --------------------------------------------------------------------------- #
# Feeder
# --------------------------------------------------------------------------- #
class Feeder:
    """IEEE-34 wrapper exposing per-bus and per-phase voltages."""

    def __init__(self, hub_buses, control_mode="OFF"):
        self.hub_buses = list(hub_buses)
        self.control_mode = control_mode
        dss.Command("Clear")
        dss.Command(f'Compile "{MASTER}"')
        dss.Command(f"Set ControlMode={control_mode}")
        dss.Command("Set MaxIterations=100")
        dss.Command("CalcVoltageBases")
        dss.Command("Solve")
        self.buses = [b for b in dss.Circuit.AllBusNames() if b.lower() != "sourcebus"]
        self.hub_kv = {}
        for b in self.hub_buses:
            dss.Circuit.SetActiveBus(b)
            kvb = dss.Bus.kVBase()
            kv = round(kvb * np.sqrt(3), 3) if kvb > 0.1 else BASE_LL_KV.get(b.lower(), 24.9)
            self.hub_kv[b] = kv
            dss.Command(f"New Generator.hub{b} bus1={b}.1.2.3 phases=3 kv={kv} "
                        f"kw=0 kvar=0 model=1 Vminpu=0.5 Vmaxpu=1.5 status=fixed")
        dss.Command("Solve")

    # ---- actuation ----
    def set_load(self, lam):
        dss.Command(f"Set LoadMult={lam}")

    def set_hub(self, bus, p_kw, q_kvar):
        dss.Command(f"Generator.hub{bus}.kW={p_kw}")
        dss.Command(f"Generator.hub{bus}.kvar={q_kvar}")

    def zero_hubs(self):
        for b in self.hub_buses:
            self.set_hub(b, 0.0, 0.0)

    def solve(self):
        dss.Command("Solve")
        return dss.Solution.Converged()

    # ---- measurement ----
    def phase_vpu(self):
        """Every energized node (per-phase) voltage in p.u. — the ANSI-relevant set."""
        out = []
        for b in self.buses:
            dss.Circuit.SetActiveBus(b)
            out.extend(v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01)
        return np.asarray(out)

    def bus_vpu(self):
        """Per-bus voltage, averaged over that bus's energized phases."""
        out = np.empty(len(self.buses))
        for i, b in enumerate(self.buses):
            dss.Circuit.SetActiveBus(b)
            vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
            out[i] = np.mean(vs) if vs else 1.0
        return out

    def hub_vpu(self, bus):
        dss.Circuit.SetActiveBus(bus)
        vs = [v for v in dss.Bus.puVmagAngle()[0::2] if v > 0.01]
        return float(np.mean(vs)) if vs else 1.0

    def feeder_mean(self):
        return float(np.mean(self.bus_vpu()))

    def tap_positions(self):
        """Regulator tap positions — non-trivial only when control_mode=STATIC."""
        taps = []
        names = dss.RegControls.AllNames()
        if not names or names == ['NONE']:
            return taps
        for n in names:
            dss.RegControls.Name(n)
            taps.append(dss.RegControls.TapNumber())
        return taps


# --------------------------------------------------------------------------- #
# Droop baseline — paper-stated deadband and saturation
# --------------------------------------------------------------------------- #
def droop_pq(v, P_rated, Q_rated, db=0.02, sat_lo=0.90, sat_hi=1.10):
    """Piecewise-linear Volt-Watt / Volt-Var. Positive P = discharge (support)."""
    if v < 1 - db:
        f = min(1.0, (1 - db - v) / (1 - db - sat_lo))
    elif v > 1 + db:
        f = -min(1.0, (v - (1 + db)) / (sat_hi - (1 + db)))
    else:
        f = 0.0
    return f * P_rated, f * Q_rated


# --------------------------------------------------------------------------- #
# EV fleet
# --------------------------------------------------------------------------- #
class EVFleet:
    """Aggregate hub fleet: availability-limited power, SOC state, throughput log."""

    def __init__(self, cfg=CFG, avail_scale=1.0, soc_on="S"):
        """soc_on: "S" drains the battery on APPARENT power, per the paper's Eq. (4)
        (P_fleet = S_req / eta_inv). "P" drains on real power only -- which leaves
        reactive support free and unlimited, and lets a learned policy hold voltage
        with Q at zero SOC cost. Kept as a switch so the sensitivity can be reported."""
        self.c = cfg
        self.avail_scale = avail_scale
        self.soc_on = soc_on
        self.reset()

    def reset(self, n_avail_day=None, soc_init=None):
        """n_avail_day: length-24 int array of available EVs (paired scenario draw)."""
        self.soc = self.c["soc_init"] if soc_init is None else float(soc_init)
        self.soh = self.c["soh"]
        self.throughput = 0.0          # cumulative battery kWh moved
        self.soc_series = [self.soc]
        self._n_day = n_avail_day

    def n_avail(self, hour):
        if self._n_day is not None:
            return int(self._n_day[hour])
        frac = np.clip(AVAIL_MEAN[hour] * self.avail_scale, 0.05, 1.0)
        return max(1, int(round(self.c["n_ev"] * frac)))

    def avail_power(self, hour):
        """Max discharge power this 1-h step: min(C-rate limit, usable-energy limit)."""
        n = self.n_avail(hour)
        cap, soh = self.c["ev_capacity"], self.soh
        p_crate = n * self.c["c_rate"] * cap
        e_usable = n * max(0.0, self.soc - self.c["soc_min"]) * cap * soh
        return min(p_crate, e_usable)

    def apply(self, p_grid, q_grid, hour, commit=True):
        """Scale requested (p, q) to fleet capability. Returns (p_sup, q_sup, rho, n)."""
        s_req = float(np.hypot(p_grid, q_grid))
        p_fleet = s_req / self.c["eta_inv"]
        p_avail = self.avail_power(hour)
        rho = min(1.0, p_avail / p_fleet) if p_fleet > 1e-6 else 1.0
        p_sup, q_sup = rho * p_grid, rho * q_grid
        if not commit:
            return p_sup, q_sup, rho, self.n_avail(hour)

        n = self.n_avail(hour)
        cap_tot = n * self.c["ev_capacity"] * self.soh
        eta = self.c["eta_inv"]
        s_sup = float(np.hypot(p_sup, q_sup))
        if p_sup >= 0:                       # net real-power export -> discharging
            e_out = (s_sup if self.soc_on == "S" else abs(p_sup)) / eta
            e_in = 0.0
        else:                                # charging; reactive support still costs
            e_in = abs(p_sup) * eta
            e_out = (abs(q_sup) / eta) if self.soc_on == "S" else 0.0
        dsoc = (e_in - e_out) / cap_tot
        self.throughput += (e_in + e_out) * 1.0            # battery kWh moved this hour
        self.soc = float(np.clip(self.soc + dsoc, self.c["soc_min"], self.c["soc_max"]))
        self.soc_series.append(self.soc)
        return p_sup, q_sup, rho, n


def draw_availability(rng, n_ev, avail_scale=1.0):
    """Binomial availability realisation for one day — the paired scenario primitive."""
    p = np.clip(AVAIL_MEAN * avail_scale, 0.02, 1.0)
    return np.maximum(1, rng.binomial(n_ev, p))


# --------------------------------------------------------------------------- #
# Voltage reward (paper Eqs. 8-10) — unchanged, so the baseline is comparable
# --------------------------------------------------------------------------- #
def reward_from_v(vpu, vmin=0.95, vmax=1.05):
    inb = bool(np.all((vpu >= vmin) & (vpu <= vmax)))
    Rvb = 10.0 if inb else 0.0
    pen = float(np.where(vpu < vmin, (vmin - vpu) * 100,
                np.where(vpu > vmax, (vpu - vmax) * 100, 0.0)).sum())
    return Rvb - pen


In [ ]:
%%writefile v2g_metrics.py
"""
Metric set for the V2G gap study.

Reports the paper's metric *and* the standard-compliant ones side by side, so the
two can be compared directly rather than argued about:

  ViolMean  hours where the FEEDER-MEAN voltage is out of band   <- the paper's metric
  ViolBus   hours where ANY bus (phase-averaged) is out of band
  ViolPh    hours where ANY energized phase is out of band       <- ANSI C84.1
  ViolHi    hours with an OVERvoltage specifically
  IntViol   integrated violation magnitude, p.u.-hours (= IntLo + IntHi)
  IntLo/IntHi   the under- and over-voltage halves of IntViol
  VMean/VMin/VMax   feeder-mean voltage stats, matching the paper's table columns
  VphMin/VphMax     worst single-phase voltage extremes over the day

Two properties this set is built for:

1. ALL VIOLATION METRICS ARE TWO-SIDED. A one-sided (undervoltage-only) metric scores an
   agent that shoves the feeder above 1.05 as violation-free -- which is exactly what an
   unconstrained learned policy will do when reactive power is cheap.
2. Counts saturate (every controller can tie at "all 18 hours violated"); IntViol does
   not, so it still separates controllers when the counts agree.
"""
import numpy as np

V_MIN, V_MAX = 0.95, 1.05
# Buses can sit exactly on a limit (e.g. a regulated bus pinned at 1.05 p.u.). Without a
# tolerance, floating-point noise flags every such hour as a violation while the integrated
# magnitude stays 0.0 -- a visibly self-contradictory pair. 1e-4 p.u. is far below any
# real measurement resolution.
V_TOL = 1e-4


def hourly_record():
    return {"hour": [], "vmean": [], "vbus_min": [], "vbus_max": [],
            "vph_min": [], "vph_max": [],
            "int_viol": [], "int_lo": [], "int_hi": [],
            "disch": [], "soc": [], "n_ev": [], "rho": [],
            "throughput": [], "taps": []}


def log_hour(rec, hour, feeder, disch, soc, n_ev, rho, throughput, taps=None):
    vbus = feeder.bus_vpu()
    vph = feeder.phase_vpu()
    rec["hour"].append(hour)
    rec["vmean"].append(float(np.mean(vbus)))
    rec["vbus_min"].append(float(vbus.min()))
    rec["vbus_max"].append(float(vbus.max()))
    rec["vph_min"].append(float(vph.min()))
    rec["vph_max"].append(float(vph.max()))
    # Integrated magnitude is TWO-SIDED: over- and undervoltage both count. A one-sided
    # version scores an agent that pushes the feeder above 1.05 as violation-free.
    lo = float(np.clip(V_MIN - V_TOL - vph, 0, None).sum())
    hi = float(np.clip(vph - V_MAX - V_TOL, 0, None).sum())
    rec["int_lo"].append(lo)
    rec["int_hi"].append(hi)
    rec["int_viol"].append(lo + hi)
    rec["disch"].append(float(disch))
    rec["soc"].append(float(soc))
    rec["n_ev"].append(float(n_ev))
    rec["rho"].append(float(rho))
    rec["throughput"].append(float(throughput))
    rec["taps"].append(list(taps) if taps else [])


def rainflow_depths(soc_series):
    """Compact rainflow: turning-point extraction then range counting.

    Returns the list of half-cycle depths (in SOC fraction). Used for evaluation
    only -- it is path-dependent and awkward inside an RL reward, so the reward
    uses Ah-throughput instead.
    """
    s = np.asarray(soc_series, dtype=float)
    if s.size < 3:
        return []
    # keep local extrema
    tp = [s[0]]
    for i in range(1, s.size - 1):
        if (s[i] - s[i - 1]) * (s[i + 1] - s[i]) < 0:
            tp.append(s[i])
    tp.append(s[-1])
    depths, stack = [], []
    for v in tp:
        stack.append(v)
        while len(stack) >= 3:
            a, b, c = stack[-3], stack[-2], stack[-1]
            if abs(b - a) <= abs(c - b):
                depths.append(abs(b - a))
                stack.pop(-2)
            else:
                break
    for i in range(len(stack) - 1):
        depths.append(abs(stack[i + 1] - stack[i]))
    return depths


def summarize(rec, soc_series=None):
    vm = np.asarray(rec["vmean"])
    vb, vbx = np.asarray(rec["vbus_min"]), np.asarray(rec["vbus_max"])
    vp, vpx = np.asarray(rec["vph_min"]), np.asarray(rec["vph_max"])
    depths = rainflow_depths(soc_series) if soc_series is not None else []
    n_taps = 0
    prev = None
    for t in rec["taps"]:
        if prev is not None and t and list(t) != list(prev):
            n_taps += sum(1 for x, y in zip(t, prev) if x != y)
        prev = t
    return dict(
        VMean=round(float(vm.mean()), 3),
        VMin=round(float(vm.min()), 3),
        VMax=round(float(vm.max()), 3),
        # two-sided: an hour counts if anything is outside [V_MIN, V_MAX]
        ViolMean=int(((vm < V_MIN - V_TOL) | (vm > V_MAX + V_TOL)).sum()),
        ViolBus=int(((vb < V_MIN - V_TOL) | (vbx > V_MAX + V_TOL)).sum()),
        ViolPh=int(((vp < V_MIN - V_TOL) | (vpx > V_MAX + V_TOL)).sum()),
        ViolHi=int((vpx > V_MAX + V_TOL).sum()),
        VphMin=round(float(vp.min()), 3),
        VphMax=round(float(vpx.max()), 3),
        IntViol=round(float(np.sum(rec["int_viol"])), 2),
        IntLo=round(float(np.sum(rec["int_lo"])), 2),
        IntHi=round(float(np.sum(rec["int_hi"])), 2),
        Energy=round(float(np.sum(rec["disch"])), 1),
        Thru=round(float(rec["throughput"][-1]) if rec["throughput"] else 0.0, 1),
        SOCend=round(float(rec["soc"][-1]), 3),
        MaxDoD=round(float(max(depths)) if depths else 0.0, 3),
        SumDoD=round(float(sum(depths)) if depths else 0.0, 3),
        TapOps=int(n_taps),
    )


# ---- multi-seed aggregation with paired (common-random-number) comparisons ---- #
def aggregate(rows, keys=None):
    """rows: list of summarize() dicts across seeds. Returns mean/std/CI per key."""
    keys = keys or [k for k in rows[0] if isinstance(rows[0][k], (int, float))]
    out = {}
    n = len(rows)
    for k in keys:
        v = np.array([r[k] for r in rows], dtype=float)
        sd = float(v.std(ddof=1)) if n > 1 else 0.0
        out[k] = dict(mean=float(v.mean()), std=sd,
                      ci95=1.96 * sd / np.sqrt(n) if n > 1 else 0.0)
    return out


def paired_delta(rows_a, rows_b, key):
    """Paired difference a-b on identical scenarios (CRN). Returns mean, ci95, n_wins."""
    a = np.array([r[key] for r in rows_a], dtype=float)
    b = np.array([r[key] for r in rows_b], dtype=float)
    d = a - b
    n = d.size
    sd = float(d.std(ddof=1)) if n > 1 else 0.0
    return dict(mean=float(d.mean()), ci95=1.96 * sd / np.sqrt(n) if n > 1 else 0.0,
                std=sd, n=n, a_better=int((d < 0).sum()), ties=int((d == 0).sum()))


def fmt_table(title, header, rows):
    w = [max(len(str(header[i])), *(len(str(r[i])) for r in rows)) + 2
         for i in range(len(header))]
    line = "".join(str(header[i]).rjust(w[i]) for i in range(len(header)))
    print(f"\n{title}")
    print(line)
    print("-" * len(line))
    for r in rows:
        print("".join(str(r[i]).rjust(w[i]) for i in range(len(r))))


In [ ]:
%%writefile v2g_env2.py
"""
Closed-loop training environment for the V2G gap study.

What this changes relative to the paper's two-phase setup, and why:

  * Episode = ONE DAY (hours 06:00-23:00) driven by the daily load shape, with the
    peak multiplier sampled per episode. The paper samples the load multiplier
    i.i.d. per step from [0.1, 4.0], which leaves the MDP with no temporal
    structure -- intertemporal rationing is not representable there.
  * The EV fleet is IN THE TRAINING LOOP, so SOC depletes as the agent discharges.
    The paper applies the fleet only at deployment (Phase 2) via rho-clipping.
  * SOC, availability and hour-of-day are IN THE STATE, so the agent can condition
    on how much energy it has left and how much of the day remains.
  * The reward carries an explicit Ah-throughput (degradation) term. The paper's
    reward is purely voltage (Eqs. 8-10) and has no battery term at all.

Action modes:
  "direct"    p = a * P_rated                       (the paper's formulation, Eq. 7)
  "residual"  p = clip(droop_p + a * P_rated, ...)  (droop prior; a=0 reproduces droop
                                                     at each STEP -- note this is a
                                                     per-step floor, NOT a day-level
                                                     performance guarantee, since
                                                     discharging early leaves less later)

Reward:
    r = R_vb - R_vp - w_deg * (battery kWh this step) / (P_rated_total * dt)

w_deg is expressed on the same scale as the voltage penalty (~100 per p.u. of
deviation), so sweeping it from 0 upward traces the violation/wear trade-off
directly. Sweep it rather than guessing one value.
"""
import numpy as np
import gymnasium as gym
from gymnasium import spaces

from v2g_sys import (CFG, Feeder, EVFleet, droop_pq, lam_profile,
                     reward_from_v, draw_availability)


class V2GDayEnv(gym.Env):
    metadata = {"render_modes": []}

    def __init__(self, hub_buses, peak_range=(1.2, 3.3), mode="residual",
                 w_deg=0.0, control_mode="OFF", ev_in_loop=True,
                 reward_on="bus", iid_lambda=False, seed=0, cfg=CFG,
                 state_fleet=True):
        super().__init__()
        self.cfg = cfg
        self.mode = mode
        # state_fleet=False blanks the SOC and availability entries, leaving the paper's
        # state (bus voltages + load multiplier) plus the hour. The observation VECTOR keeps
        # its width so a policy trained either way is loadable against the same space --
        # only the information content changes. This is the ablation that asks whether the
        # agent actually uses its battery state or only reacts to voltage.
        self.state_fleet = bool(state_fleet)
        # Optional callable (env, hour, {bus: (p, q)}) -> {bus: (p, q)}, applied after the
        # policy's setpoints are computed and before the fleet commits. None = passthrough,
        # which reproduces the unfiltered dynamics exactly.
        self.safety_filter = None
        self.w_deg = float(w_deg)
        self.peak_range = peak_range
        self.ev_in_loop = ev_in_loop
        self.reward_on = reward_on
        # iid_lambda=True reproduces the paper's Phase-1 training distribution: the load
        # multiplier is drawn i.i.d. per STEP from [0.1, 4.0], so the episode carries no
        # temporal structure. Training-only -- evaluation always uses the daily profile.
        self.iid_lambda = iid_lambda
        self.lam_iid_range = (0.1, 4.0)
        self.hours = cfg["active_hours"]
        self.rng = np.random.default_rng(seed)

        self.fd = Feeder(hub_buses, control_mode=control_mode)
        self.hubs = self.fd.hub_buses
        self.nh = len(self.hubs)
        self.fleets = {b: EVFleet(cfg) for b in self.hubs}

        n_obs = len(self.fd.buses) + 2 + 2 * self.nh
        self.observation_space = spaces.Box(-np.inf, np.inf, (n_obs,), np.float32)
        self.action_space = spaces.Box(-1.0, 1.0, (2 * self.nh,), np.float32)

        self.p_total = self.nh * cfg["P_rated"]
        self._t = 0
        self.peak = peak_range[0]

    # ------------------------------------------------------------------ #
    def _obs(self, vbus, lam):
        lam_norm = (lam - 0.1) / (4.0 - 0.1)
        hour_norm = self._t / max(1, len(self.hours) - 1)
        if self.state_fleet:
            socs = [self.fleets[b].soc for b in self.hubs]
            navs = [self.fleets[b].n_avail(self.hours[min(self._t, len(self.hours) - 1)])
                    / self.cfg["n_ev"] for b in self.hubs]
        else:
            socs = navs = [0.0] * self.nh
        return np.concatenate([vbus, [lam_norm, hour_norm], socs, navs]).astype(np.float32)

    def _lam_at(self, h):
        """Load multiplier for hour h. i.i.d. draw in paper-Phase-1 mode, else the profile."""
        if self.iid_lambda:
            return float(self.rng.uniform(*self.lam_iid_range))
        return float(lam_profile(self.peak)[h])

    def reset(self, *, seed=None, options=None):
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        options = options or {}
        self.peak = float(options.get("peak",
                          self.rng.uniform(*self.peak_range)))
        avail_day = options.get("avail_day")
        soc0 = options.get("soc_init")
        for b in self.hubs:
            day = avail_day if avail_day is not None else draw_availability(
                self.rng, self.cfg["n_ev"])
            self.fleets[b].reset(n_avail_day=day, soc_init=soc0)
        self._t = 0
        lam0 = self._lam_at(self.hours[0])
        self.fd.set_load(lam0); self.fd.zero_hubs(); self.fd.solve()
        return self._obs(self.fd.bus_vpu(), lam0), {}

    # ------------------------------------------------------------------ #
    def _setpoint(self, b, i, a):
        P, Q = self.cfg["P_rated"], self.cfg["Q_rated"]
        if self.mode == "residual":
            v = self.fd.hub_vpu(b)
            p_d, q_d = droop_pq(v, P, Q)
            p = float(np.clip(p_d + a[2 * i] * P, -P, P))
            q = float(np.clip(q_d + a[2 * i + 1] * Q, -Q, Q))
        else:                                   # "direct" -- the paper's Eq. 7
            p = float(np.clip(a[2 * i] * P, -P, P))
            q = float(np.clip(a[2 * i + 1] * Q, -Q, Q))
        return p, q

    def step(self, action):
        h = self.hours[self._t]
        self.fd.set_load(self._lam_at(h)); self.fd.zero_hubs(); self.fd.solve()

        thru_before = sum(self.fleets[b].throughput for b in self.hubs)
        a = np.asarray(action, dtype=float)

        # Pass 1 -- all setpoints. In residual mode _setpoint reads hub_vpu, and no solve
        # happens between hubs, so every hub reads the SAME zero-hub solve above. Computing
        # them all up front is therefore identical to computing them inline, and it lets a
        # safety layer see the whole commanded dispatch before any energy is committed.
        raw = {b: self._setpoint(b, i, a) for i, b in enumerate(self.hubs)}

        # Pass 2 -- optional projection onto the voltage-feasible set, BEFORE the fleet
        # commits SOC, so a projected-away command costs no battery energy.
        if self.safety_filter is not None:
            raw = self.safety_filter(self, h, raw)

        # Pass 3 -- commit
        p_sup_total = 0.0
        p_batt_uncapped = 0.0
        pq = {}                      # per-hub committed setpoints, for the P/Q angle study
        for b in self.hubs:
            p, q = raw[b]
            if self.ev_in_loop:
                p, q, _, _ = self.fleets[b].apply(p, q, h, commit=True)
            else:
                # fleet model disabled: no SOC/availability limit, but still account the
                # battery energy the command implies, so energy columns stay comparable.
                p_batt_uncapped += abs(p) / self.cfg["eta_inv"]
            self.fd.set_hub(b, p, q)
            pq[b] = (float(p), float(q))
            p_sup_total += max(0.0, p)
        self.fd.solve()
        thru = (sum(self.fleets[b].throughput for b in self.hubs) - thru_before
                if self.ev_in_loop else p_batt_uncapped)

        v = self.fd.phase_vpu() if self.reward_on == "phase" else self.fd.bus_vpu()
        r = reward_from_v(v, self.cfg["v_min"], self.cfg["v_max"])
        r -= self.w_deg * (thru / max(1e-6, self.p_total))

        self._t += 1
        done = self._t >= len(self.hours)
        nh = self.hours[min(self._t, len(self.hours) - 1)]
        obs = self._obs(self.fd.bus_vpu(), self._lam_at(nh))
        return obs, float(r), bool(done), False, {"thru": thru, "p_sup": p_sup_total,
                                                  "pq": pq, "hour": h}


In [ ]:
%%writefile v2g_study.py
"""
Study driver: paired scenarios, controller rollouts, and the five experiments.

Two structural rules enforced here:

1. ONE LIVE CIRCUIT. Feeder.__init__ issues Clear/Compile, which resets the global
   OpenDSS state and silently invalidates any previously built Feeder. So each
   experiment builds exactly one V2GDayEnv, trains on it, and evaluates every
   controller through env.fd / env.fleets. Never hold two feeders at once.

2. COMMON RANDOM NUMBERS. Every controller in a comparison is evaluated on the
   IDENTICAL scenario list -- same availability realisation, same initial SOC, same
   peak. Differences are then paired, which removes scenario noise from the
   comparison instead of leaving it in the error bars.
"""
import time
from collections import namedtuple

import numpy as np
import torch
torch.set_num_threads(4)

from v2g_sys import CFG, droop_pq, lam_profile, draw_availability
from v2g_env2 import V2GDayEnv
import v2g_metrics as M
from stable_baselines3 import SAC

Scenario = namedtuple("Scenario", "peak avail_day soc_init")


def make_scenarios(n, peak, seed0=0, n_ev=None, soc_init=None):
    """n paired scenarios at a fixed load peak."""
    n_ev = n_ev or CFG["n_ev"]
    out = []
    for k in range(n):
        rng = np.random.default_rng(1000 + seed0 + k)
        out.append(Scenario(peak=peak,
                            avail_day=draw_availability(rng, n_ev),
                            soc_init=soc_init if soc_init is not None else CFG["soc_init"]))
    return out


# --------------------------------------------------------------------------- #
# Controller rollouts -- all driven through the env's single live feeder
# --------------------------------------------------------------------------- #
def _reset_fleets(env, scen):
    for b in env.hubs:
        env.fleets[b].reset(n_avail_day=scen.avail_day, soc_init=scen.soc_init)


def _log(env, rec, h, disch, rho, n, thru_cum):
    """thru_cum is passed explicitly: when the fleet model is disabled the fleet objects
    never accumulate, so reading fleet.throughput would silently report zero energy."""
    soc = float(np.mean([env.fleets[b].soc for b in env.hubs]))
    M.log_hour(rec, h, env.fd, disch, soc, n, rho, thru_cum, env.fd.tap_positions())


def droop_equilibrium(env, h, ev_constrained, damp=0.3, iters=200, tol=1e-2,
                      max_backoff=4, backoff=True):
    """Damped Jacobi iteration to the droop fixed point, with a real convergence test.

    A FIXED ITERATION COUNT HIDES NON-CONVERGENCE. At damp=0.6 the five-hub system does not
    settle: it enters a period-2 limit cycle -- total active power alternating 1298 / 630 kW
    at mild-load hour 18 -- so "iters=25" returns whichever branch iteration 25 lands on
    rather than an equilibrium, and the answer flips with the parity of the iteration count.
    Measured behaviour at that hour: damp 0.2 and 0.3 converge to 922 kW; 0.5 oscillates
    1059/801; 0.6 oscillates 1298/630; 0.8 oscillates 1978/-110.

    So: iterate to a residual test rather than a fixed count, and if the residual stalls,
    halve the damping and restart. Returns (setpoints, converged, iters_used, damp_used).

    backoff=False disables the retry, so `iters` means EXACTLY that many iterations at the
    given damping. E9 needs that: with backoff on, a small `iters` silently becomes four
    restarts at successively halved damping, which is not "a partially converged loop" and
    made the sweep read non-monotone (iters=25 landing below iters=10).
    """
    cfg = env.cfg
    for _ in range(max_backoff if backoff else 1):
        sp = {b: (0.0, 0.0) for b in env.hubs}
        for k in range(iters):
            delta = 0.0
            for b in env.hubs:
                v = env.fd.hub_vpu(b)
                p, q = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
                p0, q0 = sp[b]
                pn, qn = (1 - damp) * p0 + damp * p, (1 - damp) * q0 + damp * q
                delta = max(delta, abs(pn - p0), abs(qn - q0))
                sp[b] = (pn, qn)
            for b, (p, q) in sp.items():
                if ev_constrained:
                    p, q, _, _ = env.fleets[b].apply(p, q, h, commit=False)
                env.fd.set_hub(b, p, q)
            env.fd.solve()
            if delta < tol:
                return sp, True, k + 1, damp
        damp *= 0.5                      # stalled -> back off and retry
    return sp, False, iters, damp


def rollout_static(env, scen, controller="droop", ev_constrained=True, damp=0.3, iters=200,
                   backoff=True):
    """controller in {'baseline','droop'}. Returns summarize() dict."""
    cfg = env.cfg
    lam = lam_profile(scen.peak)
    _reset_fleets(env, scen)
    rec = M.hourly_record()
    thru_cum = 0.0
    n_unconv = 0
    for h in env.hours:
        env.fd.set_load(lam[h]); env.fd.zero_hubs(); env.fd.solve()
        disch, rho, n = 0.0, 1.0, np.nan

        if controller == "baseline":
            env.fd.solve()

        else:  # closed-loop droop: fixed point driven to a residual test, not a fixed count
            sp, conv, _, _ = droop_equilibrium(env, h, ev_constrained,
                                               damp=damp, iters=iters, backoff=backoff)
            n_unconv += (not conv)
            for b, (p, q) in sp.items():
                if ev_constrained:
                    p, q, rho, n = env.fleets[b].apply(p, q, h, commit=True)
                else:
                    thru_cum += abs(p) / cfg["eta_inv"]
                env.fd.set_hub(b, p, q)
                disch += max(0.0, p)
            env.fd.solve()

        if ev_constrained:
            thru_cum = float(sum(env.fleets[b].throughput for b in env.hubs))
        _log(env, rec, h, disch, rho, n, thru_cum)

    socs = env.fleets[env.hubs[0]].soc_series
    s = M.summarize(rec, socs)
    # Surfaced, not swallowed: a droop row computed from a non-converged fixed point is not
    # an equilibrium and must not be reported as one.
    s["DroopUnconv"] = int(n_unconv)
    return s, rec


def rollout_policy(env, policy, scen, ev_constrained=True):
    """Deterministic policy rollout through the env itself (guarantees obs consistency)."""
    saved, saved_iid = env.ev_in_loop, env.iid_lambda
    env.ev_in_loop = ev_constrained
    env.iid_lambda = False        # evaluation always uses the real daily load profile
    obs, _ = env.reset(options=dict(peak=scen.peak, avail_day=scen.avail_day,
                                    soc_init=scen.soc_init))
    rec = M.hourly_record()
    thru_cum = 0.0
    for t, h in enumerate(env.hours):
        act, _ = policy.predict(obs, deterministic=True)
        obs, _, done, _, info = env.step(act)
        thru_cum += info["thru"]
        _log(env, rec, h, info["p_sup"], 1.0, np.nan, thru_cum)
        if done:
            break
    env.ev_in_loop, env.iid_lambda = saved, saved_iid
    socs = env.fleets[env.hubs[0]].soc_series
    return M.summarize(rec, socs), rec


def rollout_zero(env, scen, ev_constrained=True):
    """a=0 in residual mode == open-loop droop; the per-step floor sanity check."""
    class _Zero:
        def predict(self, obs, deterministic=True):
            return np.zeros(env.action_space.shape, dtype=np.float32), None
    return rollout_policy(env, _Zero(), scen, ev_constrained)


# --------------------------------------------------------------------------- #
# Training
# --------------------------------------------------------------------------- #
def train_on(env, steps, seed=0, chunk=5000, label=""):
    m = SAC("MlpPolicy", env, learning_rate=3e-4, batch_size=256, gamma=0.99,
            buffer_size=200_000, learning_starts=1000, tau=0.005,
            policy_kwargs=dict(net_arch=[256, 256]), device="cpu",
            seed=seed, verbose=0)
    t0, done = time.time(), 0
    while done < steps:
        n = min(chunk, steps - done)
        m.learn(total_timesteps=n, reset_num_timesteps=(done == 0), progress_bar=False)
        done += n
        print(f"      [{label}] {done}/{steps}  {time.time()-t0:.0f}s", flush=True)
    return m


def build_env(hub_buses, mode="residual", w_deg=0.0, control_mode="OFF",
              peak_range=(1.2, 3.3), reward_on="bus", iid_lambda=False, seed=0):
    return V2GDayEnv(hub_buses, peak_range=peak_range, mode=mode, w_deg=w_deg,
                     control_mode=control_mode, reward_on=reward_on,
                     iid_lambda=iid_lambda, seed=seed)


def build_paper_env(hub_buses, control_mode="OFF", seed=0):
    """The paper's Phase-1 training setup: direct action, i.i.d. load multiplier per step,
    no fleet in the loop. Used as the reproduction baseline and as E4's comparison arm."""
    env = build_env(hub_buses, mode="direct", w_deg=0.0, control_mode=control_mode,
                    peak_range=(0.1, 4.0), iid_lambda=True, seed=seed)
    env.ev_in_loop = False
    return env


# --------------------------------------------------------------------------- #
# E0 -- fidelity calibration
# --------------------------------------------------------------------------- #
def E0_calibration(n_scen=3):
    """Which (ControlMode, droop saturation) reproduces the paper's baseline fingerprint?

    Paper Table I baseline: feeder-mean Min = 0.907 (mild) / 0.807 (aggressive),
    violation hours (mean metric) = 13 / 17.
    """
    target = {"mild": dict(VMin=0.907, ViolMean=13),
              "aggr": dict(VMin=0.807, ViolMean=17)}
    rows = []
    for cm in ["OFF", "STATIC"]:
        env = build_env(CFG["hub_bus_single"], control_mode=cm)
        for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
            scens = make_scenarios(n_scen, peak, seed0=0)
            rs = [rollout_static(env, s, "baseline")[0] for s in scens]
            agg = M.aggregate(rs, ["VMean", "VMin", "ViolMean", "ViolBus", "ViolPh",
                                   "IntViol", "VphMax"])
            rows.append([cm, tag,
                         f"{agg['VMean']['mean']:.3f}",
                         f"{agg['VMin']['mean']:.3f}",
                         f"{agg['ViolMean']['mean']:.1f}",
                         f"{agg['ViolBus']['mean']:.1f}",
                         f"{agg['ViolPh']['mean']:.1f}",
                         f"{agg['IntViol']['mean']:.2f}",
                         f"{target[tag]['VMin']:.3f} / {target[tag]['ViolMean']}"])
        del env
    M.fmt_table("E0  Baseline fingerprint vs paper Table I (no V2G)",
                ["ControlMode", "load", "VMean", "VMin", "ViolMean",
                 "ViolBus", "ViolPh", "IntViol", "paper VMin/Viol"], rows)
    print("\n  Pick the ControlMode whose (VMin, ViolMean) is closest to the paper column.")
    print("  ViolBus / ViolPh / IntViol show how much the mean metric hides.")
    return rows


# --------------------------------------------------------------------------- #
# E1 -- reproduction of their Tables I and II
# --------------------------------------------------------------------------- #
def E1_reproduction(control_mode="OFF", steps=20000, n_scen=5, seed=0):
    """Their table structure, with our added metric columns."""
    out = {}
    for scope, hubs in [("single", CFG["hub_bus_single"]),
                        ("multi", CFG["hub_buses_multi"])]:
        # paper-style agent: direct action, i.i.d. lambda, no fleet in training (Phase 1)
        env = build_paper_env(hubs, control_mode=control_mode, seed=seed)
        print(f"\n  [E1/{scope}] training paper-style agent (direct, i.i.d. lambda, no fleet)")
        pol = train_on(env, steps, seed=seed, label=f"E1-{scope}")
        rows = []
        for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
            scens = make_scenarios(n_scen, peak, seed0=0)
            cases = [
                ("Baseline",        lambda s: rollout_static(env, s, "baseline")[0]),
                ("RL (no EV)",      lambda s: rollout_policy(env, pol, s, False)[0]),
                ("Droop (no EV)",   lambda s: rollout_static(env, s, "droop", False)[0]),
                ("RL (EV-constr)",  lambda s: rollout_policy(env, pol, s, True)[0]),
                ("Droop (EV-con.)", lambda s: rollout_static(env, s, "droop", True)[0]),
            ]
            for name, fn in cases:
                rs = [fn(s) for s in scens]
                a = M.aggregate(rs)
                rows.append([f"{tag}/{name}",
                             f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                             f"{a['VphMax']['mean']:.3f}",
                             f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                             f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                             f"{a['IntViol']['mean']:.2f}",
                             f"{a['Thru']['mean']:.0f}", f"{a['SOCend']['mean']:.3f}"])
                out[f"{scope}/{tag}/{name}"] = rs
        M.fmt_table(f"E1  {scope}-hub reproduction  (ControlMode={control_mode}, {n_scen} paired scenarios)",
                    ["case", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                     "ViolPh", "ViolHi", "IntViol", "Thru", "SOCend"], rows)
        del env, pol
    return out


# --------------------------------------------------------------------------- #
# E2 (C1) -- multi-hub WITH realistic fleet constraints
# --------------------------------------------------------------------------- #
def E2_multihub_constrained(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1, 2)):
    """Their explicitly untested case: multi-hub coordination under 45-85% availability."""
    out = {}
    for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
        scens = make_scenarios(n_scen, peak, seed0=0)
        rows, store = [], {}
        # controllers that need no training
        env = build_env(CFG["hub_buses_multi"], mode="residual", w_deg=0.0,
                        control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2))
        store["Baseline"] = [rollout_static(env, s, "baseline")[0] for s in scens]
        store["Droop"] = [rollout_static(env, s, "droop", True)[0] for s in scens]
        store["Droop (unconstr)"] = [rollout_static(env, s, "droop", False)[0] for s in scens]
        store["a=0 floor"] = [rollout_zero(env, s, True)[0] for s in scens]
        # trained closed-loop agent, one per seed, evaluated on the same scenarios
        rl_rows = []
        for sd in seeds:
            print(f"\n  [E2/{tag}] training closed-loop agent seed={sd}")
            pol = train_on(env, steps, seed=sd, label=f"E2-{tag}-s{sd}")
            rl_rows.append([rollout_policy(env, pol, s, True)[0] for s in scens])
            del pol
        store["RL closed-loop"] = [r for rs in rl_rows for r in rs]
        for name, rs in store.items():
            a = M.aggregate(rs)
            rows.append([name,
                         f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                         f"{a['VphMax']['mean']:.3f}",
                         f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                         f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                         f"{a['IntViol']['mean']:.2f}±{a['IntViol']['ci95']:.2f}",
                         f"{a['Thru']['mean']:.0f}", f"{a['SOCend']['mean']:.3f}"])
        M.fmt_table(f"E2  multi-hub, EV-CONSTRAINED  ({tag}, {n_scen} paired scenarios x {len(seeds)} seeds)",
                    ["controller", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                     "ViolPh", "ViolHi", "IntViol(±ci)", "Thru", "SOCend"], rows)
        # paired deltas vs droop, the statistically meaningful comparison
        for key in ["IntViol", "ViolBus", "Thru"]:
            d = M.paired_delta(store["RL closed-loop"][:n_scen], store["Droop"], key)
            print(f"    paired RL-Droop  {key:8s}: {d['mean']:+.2f} ± {d['ci95']:.2f} "
                  f"(RL better in {d['a_better']}/{d['n']}, ties {d['ties']})")
        out[tag] = store
        del env
    return out


# --------------------------------------------------------------------------- #
# E3 (C2) -- degradation-weight sweep -> violation/wear frontier
# --------------------------------------------------------------------------- #
def E3_degradation_frontier(peak, weights=(0.0, 1.0, 3.0, 10.0, 30.0, 100.0),
                            control_mode="OFF", steps=20000, n_scen=5, seed=0,
                            hubs=None):
    """Trace the trade-off. Droop is plotted as a POINT on this frontier, not a rival."""
    hubs = hubs or CFG["hub_buses_multi"]
    scens = make_scenarios(n_scen, peak, seed0=0)
    pts, rows = [], []

    env0 = build_env(hubs, mode="residual", w_deg=0.0, control_mode=control_mode,
                     peak_range=(peak * 0.8, peak * 1.2))
    dr = [rollout_static(env0, s, "droop", True)[0] for s in scens]
    a = M.aggregate(dr)
    rows.append(["droop (reference)", "-",
                 f"{a['IntViol']['mean']:.2f}", f"{a['IntHi']['mean']:.2f}",
                 f"{a['Thru']['mean']:.0f}",
                 f"{a['ViolBus']['mean']:.1f}", f"{a['VphMax']['mean']:.3f}",
                 f"{a['SOCend']['mean']:.3f}", f"{a['MaxDoD']['mean']:.3f}"])
    pts.append(dict(w="droop", IntViol=a["IntViol"]["mean"], Thru=a["Thru"]["mean"],
                    ViolBus=a["ViolBus"]["mean"]))
    del env0

    for w in weights:
        env = build_env(hubs, mode="residual", w_deg=w, control_mode=control_mode,
                        peak_range=(peak * 0.8, peak * 1.2))
        print(f"\n  [E3] training w_deg={w}")
        pol = train_on(env, steps, seed=seed, label=f"E3-w{w}")
        rs = [rollout_policy(env, pol, s, True)[0] for s in scens]
        a = M.aggregate(rs)
        rows.append([f"RL w_deg={w:g}", f"{w:g}",
                     f"{a['IntViol']['mean']:.2f}", f"{a['IntHi']['mean']:.2f}",
                     f"{a['Thru']['mean']:.0f}",
                     f"{a['ViolBus']['mean']:.1f}", f"{a['VphMax']['mean']:.3f}",
                     f"{a['SOCend']['mean']:.3f}", f"{a['MaxDoD']['mean']:.3f}"])
        pts.append(dict(w=w, IntViol=a["IntViol"]["mean"], Thru=a["Thru"]["mean"],
                        ViolBus=a["ViolBus"]["mean"]))
        del env, pol

    M.fmt_table(f"E3  violation / wear frontier  (peak={peak}, {n_scen} paired scenarios)",
                ["controller", "w_deg", "IntViol", "IntHi", "Thru(kWh)", "ViolBus",
                 "VphMax", "SOCend", "MaxDoD"], rows)
    print("\n  Read it as a frontier: IntViol should rise as Thru falls. Where droop sits")
    print("  relative to the RL frontier is the result -- above, on, or below it.")
    return pts


# --------------------------------------------------------------------------- #
# E4 (C3) -- the multi-hub aggressive stress case (their droop 2 vs RL 15)
# --------------------------------------------------------------------------- #
def E4_stress(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1, 2)):
    """Does day-structured, fleet-in-loop training change the aggressive-case gap?

    Also runs the paper-style agent (i.i.d. load multiplier, no fleet in training) on
    the same scenarios, so the two training regimes are compared directly.
    """
    peak = CFG["peak_aggr"]
    scens = make_scenarios(n_scen, peak, seed0=0)
    store, rows = {}, []

    env = build_env(CFG["hub_buses_multi"], mode="residual", w_deg=0.0,
                    control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2))
    store["Droop"] = [rollout_static(env, s, "droop", True)[0] for s in scens]
    store["Droop (unconstr)"] = [rollout_static(env, s, "droop", False)[0] for s in scens]
    for sd in seeds:
        print(f"\n  [E4] day-structured closed-loop agent seed={sd}")
        pol = train_on(env, steps, seed=sd, label=f"E4-day-s{sd}")
        store.setdefault("RL day-structured", []).extend(
            [rollout_policy(env, pol, s, True)[0] for s in scens])
        del pol
    del env

    env2 = build_paper_env(CFG["hub_buses_multi"], control_mode=control_mode)
    for sd in seeds:
        print(f"\n  [E4] paper-style agent (i.i.d. lambda, no fleet) seed={sd}")
        pol = train_on(env2, steps, seed=sd, label=f"E4-iid-s{sd}")
        store.setdefault("RL paper-style", []).extend(
            [rollout_policy(env2, pol, s, True)[0] for s in scens])
        del pol
    del env2

    for name, rs in store.items():
        a = M.aggregate(rs)
        rows.append([name,
                     f"{a['VMean']['mean']:.3f}", f"{a['VMin']['mean']:.3f}",
                     f"{a['VphMax']['mean']:.3f}",
                     f"{a['ViolMean']['mean']:.1f}", f"{a['ViolBus']['mean']:.1f}",
                     f"{a['ViolPh']['mean']:.1f}", f"{a['ViolHi']['mean']:.1f}",
                     f"{a['IntViol']['mean']:.2f}±{a['IntViol']['ci95']:.2f}",
                     f"{a['Thru']['mean']:.0f}"])
    M.fmt_table(f"E4  multi-hub AGGRESSIVE stress  ({n_scen} paired scenarios x {len(seeds)} seeds)",
                ["controller", "VMean", "VMin", "VphMax", "ViolMean", "ViolBus",
                 "ViolPh", "ViolHi", "IntViol(±ci)", "Thru"], rows)
    return store


# --------------------------------------------------------------------------- #
# E12 -- safety projection: does the gap survive once the agent stops overvolting?
# --------------------------------------------------------------------------- #
def _bisect_max_feasible(feas, iters=18):
    """Largest lam in [0,1] with feas(lam) true. ASSUMES feas(0) is true.

    Assumes monotonicity in lam -- less injection, lower voltage. That holds on this feeder
    over the range we use, but the guarantee is only that the returned point IS feasible
    (it is verified before return by the caller), not that it is the largest such point.
    """
    lo, hi = 0.0, 1.0
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if feas(mid):
            lo = mid
        else:
            hi = mid
    return lo


def make_safety_filter(mode="scale", tol=1e-3, iters=18):
    """Project commanded setpoints onto the OVERVOLTAGE-feasible set (all phases <= v_max).

    A projection layer of this kind is standard practice in learning-based Volt-VAR control
    (SAVER; model-augmented safe Volt-VAR; safety-constrained MARL) and our own runs show
    the unprojected agent needs one: it exceeds 1.05 in every configuration, reaching 1.234
    where droop stays at 1.050. Because IntViol is two-sided, part of the RL penalty IS that
    overvoltage -- so without this layer we cannot separate "allocates worse" from "breaks
    the upper bound and is charged for it".

    modes
      scale   : reduce P and Q together along the commanded ray. The plainest projection.
      shed_q  : reduce REACTIVE power first, and touch active power only if shedding all Q
                is still not enough. Motivated by our own E7 result -- at matched apparent
                power Q buys 1.25-1.49x LESS voltage than P while costing the same stored
                energy, so reactive is the right thing to give up first.
    """
    def _filter(env, h, raw):
        vmax = env.cfg["v_max"]

        def feas(sp, sq):
            for b, (p, q) in raw.items():
                env.fd.set_hub(b, p * sp, q * sq)
            if not env.fd.solve():
                return False
            return float(env.fd.phase_vpu().max()) <= vmax + tol

        if feas(1.0, 1.0):
            return raw                                   # nothing to project

        if mode == "shed_q":
            if feas(1.0, 0.0):                           # shedding Q alone is enough
                lam = _bisect_max_feasible(lambda l: feas(1.0, l), iters)
                out = {b: (p, q * lam) for b, (p, q) in raw.items()}
                return out if feas(1.0, lam) else {b: (p, 0.0) for b, (p, q) in raw.items()}
            if feas(0.0, 0.0):                           # Q gone, now scale P
                lam = _bisect_max_feasible(lambda l: feas(l, 0.0), iters)
                if feas(lam, 0.0):
                    return {b: (p * lam, 0.0) for b, (p, q) in raw.items()}
            return {b: (0.0, 0.0) for b in raw}

        if not feas(0.0, 0.0):
            # Zero injection is already over the limit, so the overvoltage is not ours to
            # fix -- back all the way off rather than pretending a projection exists.
            return {b: (0.0, 0.0) for b in raw}
        lam = _bisect_max_feasible(lambda l: feas(l, l), iters)
        return ({b: (p * lam, q * lam) for b, (p, q) in raw.items()} if feas(lam, lam)
                else {b: (0.0, 0.0) for b in raw})

    return _filter


def rollout_policy_safe(env, policy, scen, mode=None, ev_constrained=True):
    """rollout_policy with a safety projection inserted before the fleet commits."""
    saved = env.safety_filter
    env.safety_filter = make_safety_filter(mode) if mode else None
    try:
        return rollout_policy(env, policy, scen, ev_constrained)
    finally:
        env.safety_filter = saved


def E12_safety_projection(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1, 2)):
    """One training per seed, three evaluations -- raw, ray projection, reactive-first.

    The question: our agent loses to droop on IntViol while also breaking the 1.05 limit
    that droop respects. Once the overvoltage is projected away, does the gap close?
    Either answer is reportable, and both pre-empt the obvious reviewer objection.
    """
    out, rows = {}, []
    for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
        scens = make_scenarios(n_scen, peak, seed0=0)
        env = build_env(CFG["hub_buses_multi"], mode="residual", w_deg=0.0,
                        control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2))
        store = {"Droop": [rollout_static(env, s, "droop", True)[0] for s in scens]}
        for mode, name in ((None, "RL raw"), ("scale", "RL + proj (scale)"),
                           ("shed_q", "RL + proj (shed Q)")):
            store[name] = []
        for sd in seeds:
            print(f"\n  [E12/{tag}] training seed={sd}")
            pol = train_on(env, steps, seed=sd, label=f"E12-{tag}-s{sd}")
            for mode, name in ((None, "RL raw"), ("scale", "RL + proj (scale)"),
                               ("shed_q", "RL + proj (shed Q)")):
                store[name] += [rollout_policy_safe(env, pol, s, mode)[0] for s in scens]
            del pol
        for name, rs in store.items():
            a = M.aggregate(rs)
            rows.append([f"{tag}/{name}", f"{a['IntViol']['mean']:.2f}",
                         f"{a['IntViol']['ci95']:.2f}", f"{a['IntLo']['mean']:.2f}",
                         f"{a['IntHi']['mean']:.2f}", f"{a['ViolPh']['mean']:.1f}",
                         f"{a['VphMax']['mean']:.3f}", f"{a['Thru']['mean']:.0f}"])
            out[f"{tag}/{name}"] = rs
        for name in ("RL raw", "RL + proj (scale)", "RL + proj (shed Q)"):
            d = M.paired_delta(store[name], store["Droop"], "IntViol")
            print(f"    paired {name} - Droop  IntViol: {d['mean']:+.2f} +- {d['ci95']:.2f} "
                  f"({name} better in {d['a_better']}/{d['n']})")
        del env
    M.fmt_table(f"E12  safety projection  (multi-hub, fleet-constrained, {len(seeds)} seeds "
                f"x {n_scen} paired scenarios)",
                ["case", "IntViol", "+-95%", "IntLo", "IntHi", "ViolPh", "VphMax",
                 "Thru(kWh)"], rows)
    print("\n  IntHi is the overvoltage half. If projection drives IntHi to ~0 and IntViol")
    print("  still trails droop, the gap is real allocation, not a bound violation.")
    return out


# --------------------------------------------------------------------------- #
# E10 -- ablations: does the agent use what we gave it?
# --------------------------------------------------------------------------- #
def E10_ablations(control_mode="OFF", steps=20000, n_scen=5, seeds=(0, 1),
                  w_deg_on=10.0):
    """Two ablations on the multi-hub constrained case.

    fleet-in-state   : the paper's agent sees bus voltages + load multiplier only. Ours also
                       sees SOC, availability and the hour. Blanking those entries asks
                       whether the agent uses its battery state or merely reacts to voltage.
    degradation term : w_deg = 0 vs w_deg > 0, isolating the reward change from the state
                       change so the two are not confounded in the headline result.
    """
    out, rows = {}, []
    for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
        scens = make_scenarios(n_scen, peak, seed0=0)
        for st_fleet in (True, False):
            for w in (0.0, w_deg_on):
                acc = []
                for sd in seeds:
                    env = V2GDayEnv(CFG["hub_buses_multi"], mode="residual", w_deg=w,
                                    control_mode=control_mode,
                                    peak_range=(peak * 0.8, peak * 1.2),
                                    state_fleet=st_fleet, seed=sd)
                    lab = f"E10-{tag}-{'fleet' if st_fleet else 'nofleet'}-w{w:g}-s{sd}"
                    print(f"\n  [{lab}]")
                    pol = train_on(env, steps, seed=sd, label=lab)
                    acc += [rollout_policy(env, pol, s, True)[0] for s in scens]
                    del env, pol
                a = M.aggregate(acc)
                name = f"{tag}/state={'fleet' if st_fleet else 'voltage-only'}/w={w:g}"
                out[name] = acc
                rows.append([name, f"{a['IntViol']['mean']:.2f}",
                             f"{a['IntViol']['ci95']:.2f}",
                             f"{a['ViolPh']['mean']:.1f}", f"{a['Thru']['mean']:.0f}",
                             f"{a['VphMax']['mean']:.3f}", f"{a['SOCend']['mean']:.3f}"])
    M.fmt_table(f"E10  ablations  (multi-hub, fleet-constrained, {len(seeds)} seeds x "
                f"{n_scen} paired scenarios)",
                ["case", "IntViol", "+-95%", "ViolPh", "Thru(kWh)", "VphMax", "SOCend"],
                rows)
    print("\n  fleet-in-state pays only if 'state=fleet' beats 'state=voltage-only' by more")
    print("  than the confidence interval. If it does not, say so -- the paper's simpler")
    print("  state was sufficient, which is itself a reportable result.")
    return out


# --------------------------------------------------------------------------- #
# E11 -- what P/Q split does the trained policy actually choose?
# --------------------------------------------------------------------------- #
def policy_pq_angles(env, policy, scens, ev_constrained=False, s_floor=25.0):
    """Per-hour (angle, S) pairs actually commanded by the policy. Angle in degrees.

    Costs nothing extra: it reads the setpoints the policy already committed.

    ev_constrained defaults to FALSE, and that matters. Under the fleet constraint the
    battery reaches soc_min by hour 8 or 9, so from then on every hub is idle and there is
    no allocation left to measure -- a first version of this run produced 9 usable samples
    out of a possible 180. The reference it is compared against (E7b) is also computed
    without a fleet limit, so unconstrained is the like-for-like setting.

    Sign convention: atan2(Q, P) with P > 0 lies in (-90, 90]. NEGATIVE means the hub is
    ABSORBING reactive power, which is a different action from supplying it, not a smaller
    amount of it. Those samples are returned too and counted separately rather than being
    averaged in.
    """
    saved, saved_iid = env.ev_in_loop, env.iid_lambda
    env.ev_in_loop, env.iid_lambda = ev_constrained, False
    per_hour = {h: [] for h in env.hours}
    for sc in scens:
        obs, _ = env.reset(options=dict(peak=sc.peak, avail_day=sc.avail_day,
                                        soc_init=sc.soc_init))
        for _ in env.hours:
            act, _ = policy.predict(obs, deterministic=True)
            obs, _, done, _, info = env.step(act)
            for (p, q) in info["pq"].values():
                s = float(np.hypot(p, q))
                if s >= s_floor and p > 0:            # supporting, not charging
                    per_hour[info["hour"]].append(
                        (float(np.degrees(np.arctan2(q, p))), s))
            if done:
                break
    env.ev_in_loop, env.iid_lambda = saved, saved_iid
    return per_hour


def E11_policy_pq(control_mode="OFF", steps=20000, n_scen=5, seed=0):
    """Does the learned policy find the voltage-optimal P/Q split that E7b measures?

    E7b says the optimum is ~35-40 deg at half rating and shifts to ~15-25 deg at full
    rating, against a hub rating ratio of 38.7 deg. If the agent sits at the rating ratio
    regardless of loading, it has not learned the allocation -- it is just scaling a fixed
    power factor.
    """
    from v2g_ref import angle_sweep          # local: v2g_ref imports this module
    out = {}
    for tag, peak in [("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])]:
        scens = make_scenarios(n_scen, peak, seed0=0)
        env = build_env(CFG["hub_buses_multi"], mode="direct", w_deg=0.0,
                        control_mode=control_mode, peak_range=(peak * 0.8, peak * 1.2),
                        seed=seed)
        print(f"\n  [E11/{tag}] training direct-action agent")
        pol = train_on(env, steps, seed=seed, label=f"E11-{tag}")
        ang = policy_pq_angles(env, pol, scens)
        lam = lam_profile(peak)
        s_max = float(np.hypot(CFG["P_rated"], CFG["Q_rated"]))

        rows = []
        for h in env.hours:
            v = ang[h]
            if not v:
                rows.append([h, 0, "-", "-", "-", "-", "-"])
                continue
            a = np.array([x for x, _ in v])
            s = np.array([y for _, y in v])
            n_abs = int((a < 0).sum())                 # absorbing reactive, not supplying
            # S-weighted: a hub at 600 kVA should not count the same as one at 26 kVA
            a_w = float(np.average(a, weights=s))
            s_mean = float(s.mean())
            # what WAS optimal at the injection level the agent actually chose
            opt = angle_sweep(env, lam[h], s_frac=min(1.0, s_mean / s_max))
            rows.append([h, len(v), f"{a_w:.1f}", f"{a.std():.1f}", n_abs,
                         f"{s_mean:.0f}", f"{opt['best_deg']:.1f}"])
        M.fmt_table(
            f"E11  policy P/Q angle by hour -- {tag}  (deg; 0=pure P, 90=pure Q, <0=absorbing)",
            ["h", "n", "angle_Swtd", "sd", "n_absorb", "S_mean kVA", "optimal_deg"], rows)

        allv = [x for v in ang.values() for x in v]
        if allv:
            a = np.array([x for x, _ in allv]); s = np.array([y for _, y in allv])
            print(f"    samples {len(allv)} of {len(env.hours) * len(env.hubs) * len(scens)} "
                  f"possible   |   absorbing reactive in {int((a < 0).sum())}")
            print(f"    day S-weighted mean {np.average(a, weights=s):.1f} deg   |   "
                  f"hub rating ratio 38.7 deg")
            gaps = [abs(float(r[2]) - float(r[6])) for r in rows
                    if r[1] and r[2] != "-" and r[6] != "-"]
            if gaps:
                print(f"    mean |agent - optimal| = {np.mean(gaps):.1f} deg over "
                      f"{len(gaps)} hours")
                print("    A small gap means the agent found the allocation. A gap that")
                print("    tracks the rating ratio (38.7) means it is scaling a fixed")
                print("    power factor rather than allocating.")
        else:
            print("    no usable samples -- every hub stayed below the reporting floor")
        out[tag] = ang
        del env, pol
    return out


def runtime_estimate(steps, n_trainings, sec_per_20k=440, overhead=1.25):
    """sec_per_20k=440 is MEASURED on the target machine (4000 steps took 85-92 s across
    22 trainings, so ~438 s per 20k). The earlier default of 240 was taken from a faster
    box and under-predicted a full run by a factor of ~1.8. `overhead` covers the droop and
    policy rollouts, which scale with n_scen and are not free at 200 droop iterations."""
    mins = n_trainings * steps / 20000 * sec_per_20k / 60
    print(f"  ~{n_trainings} trainings x {steps} steps  ->  approx {mins:.0f} min of training")
    print(f"  with rollout overhead: approx {mins * overhead / 60:.1f} h wall clock")


In [ ]:
%%writefile v2g_ref.py
"""
Deterministic reference studies -- no RL training, no seeds, minutes to run.

Everything here is a power-flow computation, so the numbers are exact rather than
sampled. They are also PREREQUISITES for the training runs: each one changes how the
trained results should be read, so they run first.

E5  injection scan / achievable ceiling                                     (adds X4)
    Per hour, sweep total hub injection from zero to full inverter rating and record the
    worst-phase voltage. This is the ceiling: how far ANY controller could push the feeder
    that hour, whatever its control law. Without it droop is the only reference, and
    "the learner underperformed" cannot be separated from "the hour is not clearable".

E6  droop implementation variants                                           (adds X5)
    The target paper's coordinated droop reaches 2 violation-hours at aggressive multi-hub
    load; our closed-loop droop reaches 10. Droop evaluated OPEN-LOOP -- response computed
    once from the pre-injection voltage, which is lower, so the droop fraction is larger --
    injects more and should land closer to their number. Report both rather than leave the
    discrepancy unexplained.

E7  P/Q allocation                                                          (adds the P/Q study)
    Their Eq. (4) drains the battery on APPARENT power: a kVAr of reactive support costs
    exactly as much stored energy as a kW of active support. They never explore the
    consequence. Holding S fixed and varying only the P/Q split answers the question a
    fleet operator actually faces -- given a limited energy budget, where does it buy the
    most voltage?

E5 and E7 share one scan, so they are computed together and reported separately.
"""
import numpy as np

from v2g_sys import CFG, droop_pq, lam_profile
import v2g_metrics as M
from v2g_study import make_scenarios, _reset_fleets, _log, build_env


# --------------------------------------------------------------------------- #
# Shared primitive: sweep injection at one hour, in three P/Q allocations
# --------------------------------------------------------------------------- #
def injection_scan(env, lam_h, n_grid=33):
    """Sweep total per-hub apparent power 0 -> full rating at fixed load multiplier.

    All three allocations are swept over the SAME apparent-power axis, because that is
    what the battery pays for (Eq. 4). So the comparison is like-for-like in stored
    energy, and any difference between the curves is pure network sensitivity.
    """
    cfg = env.cfg
    s_max = float(np.hypot(cfg["P_rated"], cfg["Q_rated"]))       # kVA per hub
    theta = float(np.arctan2(cfg["Q_rated"], cfg["P_rated"]))
    modes = {"P": (1.0, 0.0),                      # all active
             "Q": (0.0, 1.0),                      # all reactive
             "PQ": (np.cos(theta), np.sin(theta))}  # rating-proportional

    env.fd.set_load(lam_h)
    out = {}
    for name, (cp, cq) in modes.items():
        grid = np.linspace(0.0, s_max, n_grid)
        vmin, vmax, ivio, conv = [], [], [], []
        for s in grid:
            for b in env.hubs:
                env.fd.set_hub(b, s * cp, s * cq)
            ok = bool(env.fd.solve())
            conv.append(ok)
            if not ok:
                # A non-converged solve still leaves voltages in the OpenDSS arrays, and
                # they are meaningless -- at deep-sag hours the high-injection end of the
                # sweep sits past the nose of the PV curve. Reading them anyway produced a
                # spurious ceiling and a spurious collapse. Mask instead.
                vmin.append(np.nan); vmax.append(np.nan); ivio.append(np.nan)
                continue
            vp = env.fd.phase_vpu()
            vmin.append(float(vp.min()))
            vmax.append(float(vp.max()))
            lo = float(np.clip(M.V_MIN - M.V_TOL - vp, 0, None).sum())
            hi = float(np.clip(vp - M.V_MAX - M.V_TOL, 0, None).sum())
            ivio.append(lo + hi)
        out[name] = dict(S=grid, Vmin=np.array(vmin), Vmax=np.array(vmax),
                         IntViol=np.array(ivio), Conv=np.array(conv, dtype=bool))
    env.fd.zero_hubs()
    env.fd.solve()
    return out


def _clearing_point(scan_mode):
    """Smallest CONVERGED S that puts every phase in band; None if the hour never clears."""
    iv = scan_mode["IntViol"]
    # Parentheses matter: `&` binds tighter than `<=` in Python.
    ok = np.flatnonzero(scan_mode["Conv"] & (np.nan_to_num(iv, nan=1.0) <= 1e-9))
    return float(scan_mode["S"][ok[0]]) if ok.size else None


def _best_point(scan_mode):
    """Best worst-phase voltage over CONVERGED points, and the S that reaches it.

    Returns (nan, nan) if nothing converged at this hour.
    """
    v = scan_mode["Vmin"]
    if not np.any(np.isfinite(v)):
        return float("nan"), float("nan")
    i = int(np.nanargmax(v))
    return float(v[i]), float(scan_mode["S"][i])


def _conv_frac(sc):
    """Fraction of swept points that converged, over all three allocations."""
    tot = sum(m["Conv"].size for m in sc.values())
    good = sum(int(m["Conv"].sum()) for m in sc.values())
    return good / tot if tot else 0.0


def angle_sweep(env, lam_h, s_frac=0.5, n_ang=19):
    """Voltage uplift vs P/Q split at FIXED apparent power.

    The three-mode comparison says P beats Q; it does not say where the optimum is. Sweeping
    the injection angle theta at constant S does, and constant S means constant battery
    drain (Eq. 4) -- so this is the allocation question a fleet operator actually faces:
    given the energy you are willing to spend this hour, what power factor buys the most
    voltage? theta=0 is pure active, theta=90 is pure reactive, and the hubs' rating ratio
    (500 kW / 400 kVAr) sits at 38.7 deg.
    """
    cfg = env.cfg
    s = s_frac * float(np.hypot(cfg["P_rated"], cfg["Q_rated"]))
    env.fd.set_load(lam_h)
    angs = np.linspace(0.0, np.pi / 2, n_ang)
    vmin, conv = [], []
    for th in angs:
        for b in env.hubs:
            env.fd.set_hub(b, s * np.cos(th), s * np.sin(th))
        ok = bool(env.fd.solve())
        conv.append(ok)
        vmin.append(float(env.fd.phase_vpu().min()) if ok else np.nan)
    env.fd.zero_hubs()
    env.fd.solve()
    vmin = np.array(vmin)
    deg = np.degrees(angs)
    if np.any(np.isfinite(vmin)):
        i = int(np.nanargmax(vmin))
        best_deg, best_v = float(deg[i]), float(vmin[i])
    else:
        best_deg, best_v = float("nan"), float("nan")
    return dict(deg=deg, Vmin=vmin, Conv=np.array(conv, dtype=bool),
                best_deg=best_deg, best_Vmin=best_v, S=s)


# --------------------------------------------------------------------------- #
# E5 + E7 -- achievable ceiling and P/Q allocation
# --------------------------------------------------------------------------- #
def E5_reference_and_pq(control_mode="OFF", n_grid=33, n_scen=1):
    """Per-hour achievable ceiling and P/Q allocation, for both hub configs and peaks.

    Also rolls out closed-loop droop on the same hours so the delivered injection can be
    placed against the required injection -- the whole point of having a reference.
    """
    res = {}
    for tag, hubs in (("single", CFG["hub_bus_single"]), ("multi", CFG["hub_buses_multi"])):
        env = build_env(hubs, control_mode=control_mode)       # ONE live circuit
        for pk_name, pk in (("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])):
            lam = lam_profile(pk)
            rows_ref, rows_pq = [], []
            per_hour = {}
            for h in env.hours:
                sc = injection_scan(env, lam[h], n_grid=n_grid)
                per_hour[h] = sc

                s_clear = _clearing_point(sc["PQ"])
                v0 = float(sc["PQ"]["Vmin"][0])                 # no injection (always solves)
                vbest_pq, s_at_best = _best_point(sc["PQ"])
                rows_ref.append([h, round(lam[h], 2), round(v0, 4), round(vbest_pq, 4),
                                 round(vbest_pq - v0, 4),
                                 "-" if s_clear is None else round(s_clear, 1),
                                 "no" if s_clear is None else "yes",
                                 f"{_conv_frac(sc):.0%}"])

                # P/Q at matched apparent power: compare at full rating and at half
                half = n_grid // 2
                rows_pq.append([h, round(lam[h], 2),
                                round(sc["P"]["Vmin"][half] - v0, 4),
                                round(sc["Q"]["Vmin"][half] - v0, 4),
                                round(sc["PQ"]["Vmin"][half] - v0, 4),
                                round(sc["P"]["Vmin"][-1] - v0, 4),
                                round(sc["Q"]["Vmin"][-1] - v0, 4),
                                round(sc["PQ"]["Vmin"][-1] - v0, 4)])

            # optimal P/Q split per hour, at half and full rating
            rows_ang, ang_raw = [], {}
            for h in env.hours:
                a_half = angle_sweep(env, lam[h], s_frac=0.5)
                a_full = angle_sweep(env, lam[h], s_frac=1.0)
                ang_raw[h] = (a_half, a_full)
                rows_ang.append([h, round(lam[h], 2),
                                 round(a_half["best_deg"], 1), round(a_half["best_Vmin"], 4),
                                 round(a_full["best_deg"], 1), round(a_full["best_Vmin"], 4)])

            # fleet energy ceiling at initial SOC, for context against S_clear
            fl = env.fleets[env.hubs[0]]
            fl.reset()
            avail_kw = {h: fl.avail_power(h) for h in env.hours}

            res[f"{tag}_{pk_name}"] = dict(ref=rows_ref, pq=rows_pq, ang=rows_ang,
                                           ang_raw=ang_raw, per_hour=per_hour,
                                           avail=avail_kw, n_hubs=len(env.hubs))
        del env
    return res


def report_E5(res):
    for key, d in res.items():
        M.fmt_table(
            f"E5 achievable ceiling -- {key}  ({d['n_hubs']} hub(s), PQ allocation)",
            ["h", "lam", "Vmin(0)", "Vmin(best)", "uplift", "S_clear/hub", "clearable",
             "conv"],
            d["ref"])
        n_clear = sum(1 for r in d["ref"] if r[6] == "yes")
        print(f"    clearable hours: {n_clear}/{len(d['ref'])}")
        print("    'conv' = fraction of swept injection points whose power flow converged;"
              "\n    non-converged points are excluded from Vmin(best) and S_clear.")


def report_E7(res):
    for key, d in res.items():
        M.fmt_table(
            f"E7 P/Q allocation at matched apparent power -- {key}  (voltage uplift, p.u.)",
            ["h", "lam", "P@half", "Q@half", "PQ@half", "P@full", "Q@full", "PQ@full"],
            d["pq"])
        # nan-aware: the full-rating column is the one most likely to be masked out.
        full = np.array([[r[5], r[6], r[7]] for r in d["pq"]], dtype=float)
        half = np.array([[r[2], r[3], r[4]] for r in d["pq"]], dtype=float)
        for lab, arr in (("half rating", half), ("full rating", full)):
            n_ok = int(np.isfinite(arr).all(axis=1).sum())
            if n_ok == 0:
                print(f"    {lab}: no hour converged in all three allocations")
                continue
            mp, mq, mpq = np.nanmean(arr[:, 0]), np.nanmean(arr[:, 1]), np.nanmean(arr[:, 2])
            print(f"    mean uplift at {lab} ({n_ok}/{len(arr)} hrs) -- "
                  f"P {mp:.4f}   Q {mq:.4f}   PQ {mpq:.4f}")
            if mq > 1e-6:
                print(f"      -> P delivers {mp/mq:.2f}x the uplift of Q "
                      f"per unit of battery drain")

        M.fmt_table(
            f"E7b voltage-optimal P/Q split at fixed apparent power -- {key}",
            ["h", "lam", "best_deg@half", "Vmin@half", "best_deg@full", "Vmin@full"],
            d["ang"])
        ang = np.array([[r[2], r[4]] for r in d["ang"]], dtype=float)
        rating_deg = np.degrees(np.arctan2(CFG["Q_rated"], CFG["P_rated"]))
        print(f"    hub rating ratio sits at {rating_deg:.1f} deg "
              f"(0 = pure active, 90 = pure reactive)")
        for j, lab in ((0, "half"), (1, "full")):
            if np.any(np.isfinite(ang[:, j])):
                print(f"    mean voltage-optimal split at {lab} rating: "
                      f"{np.nanmean(ang[:, j]):.1f} deg")


# --------------------------------------------------------------------------- #
# E8 -- per-hub optimized dispatch (replaces the uniform-injection ceiling)
# --------------------------------------------------------------------------- #
def _intviol_now(fd):
    vp = fd.phase_vpu()
    lo = float(np.clip(M.V_MIN - M.V_TOL - vp, 0, None).sum())
    hi = float(np.clip(vp - M.V_MAX - M.V_TOL, 0, None).sum())
    return lo + hi


def _apply_and_score(env, sp):
    """Set every hub, solve, return two-sided violation. Non-convergence scores +inf so
    the search treats it as infeasible rather than reading stale voltages."""
    for b, (p, q) in sp.items():
        env.fd.set_hub(b, p, q)
    if not env.fd.solve():
        return float("inf")
    return _intviol_now(env.fd)


def optimal_dispatch(env, lam_h, n_pass=3, n_s=9, n_th=5, refine=True):
    """Per-hub (P, Q) minimizing two-sided violation, by coordinate descent on the AC flow.

    WHY THIS EXISTS. The uniform-injection scan answers "what can all hubs do together at
    EQUAL output", which is not the achievable ceiling. With five hubs at equal output,
    lifting the far end of the feeder to 0.95 drives the near buses past 1.05, so hours
    that are clearable by an uneven dispatch score as unclearable -- and the five-hub case
    then scores worse than the single-hub case, which is impossible. Searching per hub
    removes that artifact.

    Coordinate descent rather than a gradient method because the objective is piecewise
    linear with a kink at each voltage limit, and because every evaluation is a full AC
    power flow whose derivative we do not have.
    """
    cfg = env.cfg
    s_max = float(np.hypot(cfg["P_rated"], cfg["Q_rated"]))
    env.fd.set_load(lam_h)

    sp = {b: (0.0, 0.0) for b in env.hubs}
    best = _apply_and_score(env, sp)
    s_grid = np.linspace(0.0, s_max, n_s)
    th_grid = np.linspace(0.0, np.pi / 2, n_th)
    n_solve = 1

    for _ in range(n_pass):
        improved = False
        for b in env.hubs:
            keep, loc = sp[b], best
            for s in s_grid:
                for th in th_grid:
                    sp[b] = (s * np.cos(th), s * np.sin(th))
                    val = _apply_and_score(env, sp)
                    n_solve += 1
                    if val < loc - 1e-9:
                        loc, keep = val, sp[b]
            sp[b] = keep
            if loc < best - 1e-9:
                best, improved = loc, True
        if not improved:
            break

    if refine:                       # local polish around the incumbent
        for b in env.hubs:
            p0, q0 = sp[b]
            s0, th0 = float(np.hypot(p0, q0)), float(np.arctan2(q0, p0))
            keep, loc = sp[b], best
            ds = s_max / (n_s - 1) if n_s > 1 else s_max
            for s in np.linspace(max(0.0, s0 - ds), min(s_max, s0 + ds), 7):
                for th in np.linspace(max(0.0, th0 - 0.4), min(np.pi / 2, th0 + 0.4), 7):
                    sp[b] = (s * np.cos(th), s * np.sin(th))
                    val = _apply_and_score(env, sp)
                    n_solve += 1
                    if val < loc - 1e-9:
                        loc, keep = val, sp[b]
            sp[b] = keep
            best = min(best, loc)

    for b, (p, q) in sp.items():
        env.fd.set_hub(b, p, q)
    env.fd.solve()
    vp = env.fd.phase_vpu()
    s_tot = float(sum(np.hypot(p, q) for p, q in sp.values()))
    env.fd.zero_hubs()
    env.fd.solve()
    return dict(IntViol=best, dispatch=dict(sp), S_total=s_tot, n_solve=n_solve,
                Vmin=float(vp.min()), Vmax=float(vp.max()),
                clearable=bool(best <= 1e-9))


def E8_optimal_ceiling(control_mode="OFF", n_pass=3):
    """True achievable ceiling per hour, and the value of coordinating hubs unevenly.

    Reports optimized dispatch against the uniform-injection best, so the gap IS the value
    of coordination -- independent of any controller. That is directly relevant to the
    target paper's multi-hub claim: their RL coordinates hubs, droop is purely local.
    """
    res = {}
    for tag, hubs in (("single", CFG["hub_bus_single"]), ("multi", CFG["hub_buses_multi"])):
        env = build_env(hubs, control_mode=control_mode)
        for pk_name, pk in (("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])):
            lam = lam_profile(pk)
            rows, raw = [], {}
            for h in env.hours:
                uni = injection_scan(env, lam[h], n_grid=17)
                u_best = float(np.nanmin(uni["PQ"]["IntViol"])) if np.any(
                    np.isfinite(uni["PQ"]["IntViol"])) else float("nan")
                opt = optimal_dispatch(env, lam[h], n_pass=n_pass)
                raw[h] = opt
                rows.append([h, round(lam[h], 2),
                             round(u_best, 2), round(opt["IntViol"], 2),
                             round(opt["Vmin"], 4), round(opt["Vmax"], 4),
                             round(opt["S_total"], 0),
                             "yes" if opt["clearable"] else "no"])
            res[f"{tag}_{pk_name}"] = dict(rows=rows, raw=raw, n_hubs=len(env.hubs))
        del env
    return res


def report_E8(res):
    for key, d in res.items():
        M.fmt_table(
            f"E8 optimized per-hub dispatch -- {key}  ({d['n_hubs']} hub(s))",
            ["h", "lam", "IntViol(uniform)", "IntViol(opt)", "Vmin", "Vmax",
             "S_total kVA", "clearable"], d["rows"])
        n_clear = sum(1 for r in d["rows"] if r[7] == "yes")
        uni = np.array([r[2] for r in d["rows"]], dtype=float)
        opt = np.array([r[3] for r in d["rows"]], dtype=float)
        print(f"    clearable hours (optimized): {n_clear}/{len(d['rows'])}")
        print(f"    day-total IntViol -- uniform {np.nansum(uni):.2f}   "
              f"optimized {np.nansum(opt):.2f}")
        if d["n_hubs"] > 1 and np.nansum(uni) > 1e-9:
            gain = 1 - np.nansum(opt) / np.nansum(uni)
            print(f"    value of uneven hub coordination: {gain:.1%} lower violation "
                  f"at equal inverter rating")


# --------------------------------------------------------------------------- #
# E6 -- droop implementation variants
# --------------------------------------------------------------------------- #
def rollout_droop_openloop(env, scen, ev_constrained=True):
    """Droop computed ONCE from the pre-injection voltage, applied, solved.

    This is the natural reading of a 'local Volt-Var/Volt-Watt droop controller' if it is
    not iterated to equilibrium. The pre-injection voltage is the lowest voltage of the
    hour, so the droop fraction is at its largest -- this variant injects strictly more
    than the closed-loop version and should sit closer to the paper's Table II.
    """
    cfg = env.cfg
    lam = lam_profile(scen.peak)
    _reset_fleets(env, scen)
    rec = M.hourly_record()
    thru_cum = 0.0
    for h in env.hours:
        env.fd.set_load(lam[h])
        env.fd.zero_hubs()
        env.fd.solve()                       # pre-injection state -> droop reads this
        disch, rho, n = 0.0, 1.0, np.nan
        for b in env.hubs:
            v = env.fd.hub_vpu(b)
            p, q = droop_pq(v, cfg["P_rated"], cfg["Q_rated"])
            if ev_constrained:
                p, q, rho, n = env.fleets[b].apply(p, q, h, commit=True)
            else:
                thru_cum += abs(p) / cfg["eta_inv"]
            env.fd.set_hub(b, p, q)
            disch += max(0.0, p)
        env.fd.solve()
        if ev_constrained:
            thru_cum = float(sum(env.fleets[b].throughput for b in env.hubs))
        _log(env, rec, h, disch, rho, n, thru_cum)
    return M.summarize(rec, env.fleets[env.hubs[0]].soc_series), rec


def E6_droop_variants(control_mode="OFF", n_scen=3):
    """Open-loop vs closed-loop droop, both hub configs, both peaks, both constraint states.

    The paper's Table II coordinated-droop row is the target: VMean 1.024/0.998,
    VMin 1.004/0.940, violation-hours 0/2 for mild/aggressive.
    """
    from v2g_study import rollout_static
    out = {}
    for tag, hubs in (("single", CFG["hub_bus_single"]), ("multi", CFG["hub_buses_multi"])):
        env = build_env(hubs, control_mode=control_mode)
        for pk_name, pk in (("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])):
            scens = make_scenarios(n_scen, pk, seed0=0)
            rows = []
            for con in (False, True):
                lab = "constrained" if con else "unconstrained"
                for name, fn in (("closed-loop", rollout_static),
                                 ("open-loop", rollout_droop_openloop)):
                    accum = []
                    for sc in scens:
                        if fn is rollout_static:
                            s, _ = fn(env, sc, controller="droop", ev_constrained=con)
                        else:
                            s, _ = fn(env, sc, ev_constrained=con)
                        accum.append(s)
                    a = M.aggregate(accum)
                    rows.append([f"{name} ({lab})",
                                 round(a["VMean"]["mean"], 3), round(a["VMin"]["mean"], 3),
                                 round(a["VMax"]["mean"], 3),
                                 round(a["ViolMean"]["mean"], 1),
                                 round(a["ViolPh"]["mean"], 1),
                                 round(a["IntViol"]["mean"], 2),
                                 round(a["VphMax"]["mean"], 3),
                                 round(a["Thru"]["mean"], 0)])
            out[f"{tag}_{pk_name}"] = rows
        del env
    return out


def report_E6(out):
    paper = {"multi_mild": "paper Table II: VMean 1.024  VMin 1.004  ViolMean 0",
             "multi_aggr": "paper Table II: VMean 0.998  VMin 0.940  ViolMean 2"}
    for key, rows in out.items():
        M.fmt_table(f"E6 droop variants -- {key}",
                    ["variant", "VMean", "VMin", "VMax", "ViolMean", "ViolPh",
                     "IntViol", "VphMax", "Thru"], rows)
        if key in paper:
            print(f"    {paper[key]}")


# --------------------------------------------------------------------------- #
# E9 -- droop convergence sweep
# --------------------------------------------------------------------------- #
def E9_droop_iters(control_mode="OFF", n_scen=3,
                   iters=(1, 2, 3, 5, 10, 25, 50), damps=(0.3, 0.6)):
    """How far the droop loop is driven, swept.

    E6 showed neither pure variant reproduces both of the paper's Table II rows: closed-loop
    matches mild (VMean 1.009 vs their 1.024, ViolMean 0 vs 0) while open-loop is wildly
    over-injected there (VphMax 2.095); open-loop is closer at aggressive (1.011/0.944 vs
    their 0.998/0.940) while closed-loop undershoots (0.961/0.925).

    A PARTIALLY converged loop sits between the two. With damp=0.6 from a zero start, one
    iteration injects 0.6x the open-loop response and the sequence then decays toward the
    equilibrium, so this sweep spans the gap. If a single iteration count reproduces both
    rows, that identifies their droop; if none does, the difference is elsewhere and the
    paper reports it as a limitation rather than guessing.
    """
    from v2g_study import rollout_static
    out = {}
    for tag, hubs in (("multi", CFG["hub_buses_multi"]),
                      ("single", CFG["hub_bus_single"])):
        env = build_env(hubs, control_mode=control_mode)
        for pk_name, pk in (("mild", CFG["peak_mild"]), ("aggr", CFG["peak_aggr"])):
            scens = make_scenarios(n_scen, pk, seed0=0)
            rows = []
            for dp in damps:
                for it in iters:
                    # backoff=False so `iters=k` means exactly k iterations at damping dp.
                    # With backoff on, a small k silently becomes four restarts at halved
                    # damping, which is not a partially converged loop.
                    acc = [rollout_static(env, sc, controller="droop", ev_constrained=False,
                                          damp=dp, iters=it, backoff=False)[0]
                           for sc in scens]
                    a = M.aggregate(acc)
                    rows.append([f"d{dp}/i{it}", round(a["VMean"]["mean"], 3),
                                 round(a["VMin"]["mean"], 3),
                                 round(a["ViolMean"]["mean"], 1),
                                 round(a["ViolPh"]["mean"], 1),
                                 round(a["IntViol"]["mean"], 2),
                                 round(a["VphMax"]["mean"], 3),
                                 round(a["Thru"]["mean"], 0)])
            acc = [rollout_droop_openloop(env, sc, ev_constrained=False)[0] for sc in scens]
            a = M.aggregate(acc)
            rows.append(["open-loop", round(a["VMean"]["mean"], 3),
                         round(a["VMin"]["mean"], 3), round(a["ViolMean"]["mean"], 1),
                         round(a["ViolPh"]["mean"], 1), round(a["IntViol"]["mean"], 2),
                         round(a["VphMax"]["mean"], 3), round(a["Thru"]["mean"], 0)])
            out[f"{tag}_{pk_name}"] = rows
        del env
    return out


def report_E9(out):
    paper = {"multi_mild": (1.024, 1.004, 0.0), "multi_aggr": (0.998, 0.940, 2.0)}
    for key, rows in out.items():
        M.fmt_table(f"E9 droop convergence sweep -- {key}  (unconstrained)",
                    ["setting", "VMean", "VMin", "ViolMean", "ViolPh", "IntViol",
                     "VphMax", "Thru"], rows)
        if key not in paper:
            continue
        pm, pv, pviol = paper[key]
        print(f"    paper Table II: VMean {pm}  VMin {pv}  ViolMean {pviol:.0f}")
        # closest row on the two voltage columns the paper actually reports
        d = sorted((abs(r[1] - pm) + abs(r[2] - pv), r[0]) for r in rows)
        print(f"    closest on (VMean, VMin): {d[0][1]}  |dVMean|+|dVMin| = {d[0][0]:.3f}")
        # Is the sweep even converging? Compare the two largest iteration counts per damping.
        for dp in sorted({r[0].split("/")[0] for r in rows if "/" in r[0]}):
            th = [(int(r[0].split("i")[1]), r[7]) for r in rows if r[0].startswith(dp + "/")]
            th.sort()
            if len(th) >= 2:
                spread = abs(th[-1][1] - th[-2][1])
                verdict = "converged" if spread < 0.02 * max(1.0, abs(th[-1][1])) \
                    else "NOT converged (limit cycle)"
                print(f"    {dp}: throughput at i={th[-2][0]} vs i={th[-1][0]} -> "
                      f"{th[-2][1]:.0f} vs {th[-1][1]:.0f} kWh  [{verdict}]")


In [ ]:
# run parameters
QUICK = False        # E12 is short even at full settings; QUICK is only for a wiring check

if QUICK:
    STEPS, N_SCEN, SEEDS = 2000, 2, (0,)
else:
    STEPS, N_SCEN, SEEDS = 20000, 5, (0, 1, 2)

CONTROL_MODE = "OFF"

import importlib, numpy as np
import v2g_sys, v2g_metrics, v2g_env2, v2g_study, v2g_ref
for m in (v2g_sys, v2g_metrics, v2g_env2, v2g_study, v2g_ref):
    importlib.reload(m)
import v2g_study as S
from v2g_sys import CFG
RESULTS = {}

# 2 peaks x len(SEEDS) trainings; each policy is evaluated three ways, which needs no
# extra training -- the projection happens at rollout time.
S.runtime_estimate(STEPS, 2 * len(SEEDS))

## Sanity check — does the projection actually bind?

Before spending the training budget, confirm the filter does what it claims on an untrained
policy: `IntHi` should go to zero and `VphMax` should land on 1.050. If it does not, stop —
something is wrong and the full run would be wasted.

In [ ]:
from stable_baselines3 import SAC
_env = S.build_env(CFG["hub_buses_multi"], control_mode=CONTROL_MODE, seed=0)
_pol = S.train_on(_env, 600, seed=0, label="sanity")
_K = ("IntViol", "IntLo", "IntHi", "ViolPh", "VphMax", "Thru")
print(f"{'peak':6s} {'mode':8s} " + "  ".join(f"{k:>8s}" for k in _K))
_ok = True
for _pk, _nm in ((CFG["peak_mild"], "mild"), (CFG["peak_aggr"], "aggr")):
    _s = S.make_scenarios(1, _pk, seed0=0)[0]
    for _m in (None, "scale", "shed_q"):
        _r, _ = S.rollout_policy_safe(_env, _pol, _s, _m)
        print(f"{_nm:6s} {str(_m):8s} " + "  ".join(f"{_r[k]:8.3f}" for k in _K))
        if _m is not None and _r["IntHi"] > 1e-6:
            _ok = False
print("\nSANITY:", "PASS -- projection removes all overvoltage" if _ok
      else "FAIL -- projection left overvoltage; do not run the rest")
del _env, _pol

## E12

One training per seed, three evaluations of that same policy — raw, ray projection, and
reactive-first projection — against droop on identical paired scenarios.

In [ ]:
RESULTS["E12"] = S.E12_safety_projection(control_mode=CONTROL_MODE, steps=STEPS,
                                         n_scen=N_SCEN, seeds=SEEDS)

## What to send back

The sanity-check output and the E12 table, including the paired-delta lines printed under
each load level.

The question it answers: **once the overvoltage is projected away, does the droop-vs-RL gap
survive?** If `IntHi` goes to zero and `IntViol` still trails droop, the gap is genuine
allocation rather than a bound violation — which makes the null far more defensible. If it
closes, the paper gets materially stronger.